
# BNCI IV-2a — Hybrid Best-of-Modules v2

This notebook consolidates the strongest ideas from the uploaded Module 1–7 pipeline and the hybrid NF-EEG specification.

## Preserved strengths
- Full 9-subject LOSO.
- Train-only normalization / feature fitting.
- Paper-style OVR CSP + LASSO branch.
- EEGNet + center-loss warmup.
- Multi-scale raw EEG CNN.
- Transformer relational encoder.
- Optional BiLSTM, retained only as an ablation.
- Calibrated probability fusion.
- Temperature scaling.
- Conservative target-session fine-tuning.
- Class-wise FBGAN in a separate target-adaptation protocol.
- Synthetic EEG quality control.
- Strict separation of zero-calibration LOSO from target-adaptation results.

## Critical protocol correction
The uploaded pipeline applies Euclidean Alignment to the entire held-out subject before separating calibration and test sessions. That allows the held-out test-session covariance to influence preprocessing. In this v2 notebook:

**STRICT UNSEEN-SUBJECT LOSO**
- no target covariance
- no target calibration data
- no target FBGAN
- no target-derived ensemble weights
- no target-derived temperature

**TARGET-ADAPTATION LOSO**
- calibration session S0 is explicitly allowed
- S0-only EA may be fitted
- S0-only CSP/LASSO and FBGAN are allowed
- S1 remains untouched until final evaluation


In [1]:

# CELL 1 — IMPORTS + CONFIG
import os, gc, re, copy, time, math, random, pickle, warnings, json
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy.linalg import eigh
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, log_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

@dataclass
class CFG:
    data_root: str = "/Users/ashokvarmabevara/Project2/BCI IV-2a"
    results_dir: str = "./results_hybrid_v2"
    checkpoint_dir: str = "./checkpoints_hybrid_v2"
    seed: int = 20260824

    sfreq: int = 250
    n_channels: int = 22
    n_times: int = 1000
    n_classes: int = 4
    class_names: tuple = ("Left Hand", "Right Hand", "Both Feet", "Tongue")

    # Primary protocol
    protocol: str = "strict_loso"  # strict_loso | target_adaptation

    # Preprocessing
    main_filter: str = "none"  # none | 1_38 | 8_30
    normalize_mode: str = "channel"  # global | channel

    # Leakage-safe source-domain alignment
    use_source_whitening: bool = False

    # Neural architecture
    use_eegnet: bool = True
    use_multiscale: bool = True
    use_transformer: bool = True
    use_bilstm: bool = False
    attention: str = "se"

    patch_spatial: int = 4
    patch_temporal: int = 25
    patch_stride_spatial: int = 4
    patch_stride_temporal: int = 25

    cnn_dim: int = 64
    transformer_dim: int = 128
    transformer_heads: int = 4
    transformer_layers: int = 4
    transformer_ff: int = 256
    bilstm_hidden: int = 128
    bilstm_layers: int = 2

    fusion_dim: int = 128
    dropout: float = 0.30
    activation: str = "gelu"

    # EEGNet
    eegnet_f1: int = 8
    eegnet_d: int = 2
    eegnet_f2: int = 16

    # CSP
    csp_bands: tuple = (
        (1,4),(4,8),(8,12),(12,16),(16,20),
        (20,24),(24,28),(28,32),(32,35),(35,38)
    )
    csp_components: int = 4
    use_lasso: bool = True

    # Training
    batch_size: int = 64
    epochs: int = 100
    patience: int = 15
    lr: float = 8e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    center_lambda: float = 0.05
    center_warmup: int = 20

    # Ensemble calibration
    temperature_grid: tuple = tuple(np.arange(0.7, 3.01, 0.1))

    # Inner source validation
    n_inner_val_subjects: int = 2

    # Target adaptation
    target_use_ea: bool = True
    target_ft_epochs: int = 12
    target_ft_lr: float = 5e-5

    # FBGAN
    use_fbgan: bool = False
    gan_noise_dim: int = 1600
    gan_epochs: int = 100
    gan_batch: int = 8
    gan_lr_g: float = 2e-4
    gan_lr_d: float = 1e-4
    gan_fake_per_class: int = 250
    gan_aug_ratio: float = 0.30

    fast_dev_run: bool = False

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)

print(asdict(cfg))


{'data_root': '/Users/ashokvarmabevara/Project2/BCI IV-2a', 'results_dir': './results_hybrid_v2', 'checkpoint_dir': './checkpoints_hybrid_v2', 'seed': 20260824, 'sfreq': 250, 'n_channels': 22, 'n_times': 1000, 'n_classes': 4, 'class_names': ('Left Hand', 'Right Hand', 'Both Feet', 'Tongue'), 'protocol': 'strict_loso', 'main_filter': 'none', 'normalize_mode': 'channel', 'use_source_whitening': False, 'use_eegnet': True, 'use_multiscale': True, 'use_transformer': True, 'use_bilstm': False, 'attention': 'se', 'patch_spatial': 4, 'patch_temporal': 25, 'patch_stride_spatial': 4, 'patch_stride_temporal': 25, 'cnn_dim': 64, 'transformer_dim': 128, 'transformer_heads': 4, 'transformer_layers': 4, 'transformer_ff': 256, 'bilstm_hidden': 128, 'bilstm_layers': 2, 'fusion_dim': 128, 'dropout': 0.3, 'activation': 'gelu', 'eegnet_f1': 8, 'eegnet_d': 2, 'eegnet_f2': 16, 'csp_bands': ((1, 4), (4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 35), (35, 38)), 'csp_components': 4, '

In [2]:

# CELL 2 — REPRODUCIBILITY + DEVICE
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Torch:", torch.__version__)
print("Device:", DEVICE)
print("MPS available:", torch.backends.mps.is_available())


Torch: 2.10.0
Device: mps
MPS available: True


In [4]:
# ============================================================
# CELL 3 — ROBUST BNCI IV-2a GDF LOADER
# ============================================================

import mne
import re
from pathlib import Path
import numpy as np

from mne.io import read_raw_gdf


# ============================================================
# SUBJECT / SESSION IDENTIFICATION
# ============================================================

def infer_subject(path):
    """
    Extract A01 ... A09 from filenames such as:

        A01T.gdf
        A02T.gdf
        A09T.gdf

    IMPORTANT:
    This function must return A01, not A.
    """

    filename = Path(path).name.upper()

    match = re.search(r"A\d{2}", filename)

    if match is None:
        raise ValueError(
            f"Could not identify subject ID from filename:\n"
            f"{filename}"
        )

    return match.group(0)


def infer_session(path):
    """
    Identify the BNCI IV-2a session from the filename.

    Typical files:
        A01T.gdf -> training/session T
        A01E.gdf -> evaluation/session E

    We keep the original filename semantics but normalize
    them to S0 / S1 internally.
    """

    filename = Path(path).name.upper()

    if re.search(r"A\d{2}T\.GDF$", filename):
        return "S0"

    if re.search(r"A\d{2}E\.GDF$", filename):
        return "S1"

    # Fallback for unusual naming
    if filename.endswith("T.GDF"):
        return "S0"

    if filename.endswith("E.GDF"):
        return "S1"

    return "UNKNOWN"


# ============================================================
# EVENT MAP
# ============================================================

EVENT_MAP = {
    769: 0,   # Left hand
    770: 1,   # Right hand
    771: 2,   # Both feet
    772: 3    # Tongue
}


# ============================================================
# SINGLE GDF LOADER
# ============================================================

def load_one_gdf(path, cfg):

    raw = read_raw_gdf(
        path,
        preload=True,
        verbose=False
    )

    # --------------------------------------------------------
    # EEG channels only
    # --------------------------------------------------------

    eeg_picks = mne.pick_types(
        raw.info,
        eeg=True,
        eog=False,
        stim=False,
        exclude="bads"
    )

    if len(eeg_picks) < cfg.n_channels:

        raise RuntimeError(
            f"{Path(path).name}: "
            f"only {len(eeg_picks)} EEG channels found."
        )

    # Exactly first 22 EEG channels
    eeg_picks = eeg_picks[:cfg.n_channels]

    raw.pick(eeg_picks)

    # --------------------------------------------------------
    # Subject / session
    # --------------------------------------------------------

    sid = infer_subject(path)
    session = infer_session(path)

    # --------------------------------------------------------
    # Events
    # --------------------------------------------------------

    events, event_id = mne.events_from_annotations(
        raw,
        verbose=False
    )

    reverse = {}

    for name, code in event_id.items():

        numbers = re.findall(
            r"\d+",
            str(name)
        )

        if numbers:
            reverse[code] = int(
                numbers[-1]
            )

    # --------------------------------------------------------
    # Extract trials
    # --------------------------------------------------------

    Xs = []
    ys = []
    sessions = []

    for event in events:

        onset, _, internal_code = event

        true_code = reverse.get(
            internal_code,
            internal_code
        )

        if true_code not in EVENT_MAP:
            continue

        label = EVENT_MAP[
            true_code
        ]

        start = int(onset)
        stop = start + cfg.n_times

        if stop > raw.n_times:
            continue

        x = raw.get_data(
            start=start,
            stop=stop
        ) * 1e6

        if x.shape != (
            cfg.n_channels,
            cfg.n_times
        ):
            continue

        Xs.append(
            x.astype(np.float32)
        )

        ys.append(
            label
        )

        sessions.append(
            session
        )

    # --------------------------------------------------------
    # Empty file protection
    # --------------------------------------------------------

    if len(Xs) == 0:

        return (
            np.empty(
                (
                    0,
                    cfg.n_channels,
                    cfg.n_times
                ),
                dtype=np.float32
            ),

            np.empty(
                (0,),
                dtype=np.int64
            ),

            np.empty(
                (0,),
                dtype=str
            ),

            sid
        )

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return (
        np.stack(Xs).astype(np.float32),
        np.asarray(
            ys,
            dtype=np.int64
        ),
        np.asarray(
            sessions,
            dtype=str
        ),
        sid
    )


# ============================================================
# DISCOVER ALL GDF FILES
# ============================================================

gdf_files = sorted(
    str(p)
    for p in Path(
        cfg.data_root
    ).rglob("*.gdf")
)

if not gdf_files:

    raise FileNotFoundError(
        f"No GDF files found under:\n"
        f"{cfg.data_root}"
    )


print("=" * 80)
print("DISCOVERED GDF FILES")
print("=" * 80)

for f in gdf_files:
    print(
        Path(f).name,
        "-> subject:",
        infer_subject(f),
        "| session:",
        infer_session(f)
    )


# ============================================================
# LOAD ALL FILES
# ============================================================

X_parts = []
y_parts = []
subject_parts = []
session_parts = []


for path in gdf_files:

    Xi, yi, sesi, sid = load_one_gdf(
        path,
        cfg
    )

    if len(yi) == 0:

        print(
            f"WARNING: no MI trials in "
            f"{Path(path).name}"
        )

        continue

    X_parts.append(Xi)

    y_parts.append(yi)

    subject_parts.append(
        np.full(
            len(yi),
            sid,
            dtype=str
        )
    )

    session_parts.append(
        sesi
    )

    print(
        f"{Path(path).name}: "
        f"X={Xi.shape}, "
        f"classes={np.bincount(yi, minlength=4)}, "
        f"subject={sid}, "
        f"session={np.unique(sesi).tolist()}"
    )


# ============================================================
# CONCATENATE
# ============================================================

if not X_parts:

    raise RuntimeError(
        "No valid EEG trials were loaded."
    )


X = np.concatenate(
    X_parts,
    axis=0
)

y = np.concatenate(
    y_parts,
    axis=0
)

subjects = np.concatenate(
    subject_parts,
    axis=0
)

sessions = np.concatenate(
    session_parts,
    axis=0
)


# ============================================================
# DATA VALIDATION
# ============================================================

assert X.ndim == 3

assert X.shape[1] == cfg.n_channels

assert X.shape[2] == cfg.n_times

assert len(X) == len(y)

assert len(X) == len(subjects)

assert len(X) == len(sessions)


# ============================================================
# FINAL DATASET REPORT
# ============================================================

print("\n")
print("=" * 80)
print("DATASET")
print("=" * 80)

print(
    "X:",
    X.shape
)

print(
    "y:",
    y.shape
)

print(
    "subjects:",
    np.unique(subjects).tolist()
)

print(
    "sessions:",
    np.unique(sessions).tolist()
)


# ============================================================
# SUBJECT DISTRIBUTION
# ============================================================

print("\nSubject distribution:")

unique_subjects, subject_counts = np.unique(
    subjects,
    return_counts=True
)

for sid, count in zip(
    unique_subjects,
    subject_counts
):

    print(
        f"  {sid}: {count} trials"
    )


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print("\nClass distribution:")

for c in range(
    cfg.n_classes
):

    print(
        f"  Class {c}: "
        f"{int(np.sum(y == c))}"
    )


# ============================================================
# SESSION DISTRIBUTION
# ============================================================

print("\nSession distribution:")

unique_sessions, session_counts = np.unique(
    sessions,
    return_counts=True
)

for s, count in zip(
    unique_sessions,
    session_counts
):

    print(
        f"  {s}: {count} trials"
    )


# ============================================================
# SAFETY CHECK
# ============================================================

print("\nSafety checks:")

print(
    "Trial count:",
    len(X)
)

print(
    "Labels:",
    len(y)
)

print(
    "Subjects:",
    len(subjects)
)

print(
    "Sessions:",
    len(sessions)
)

print(
    "All aligned:",
    len(X) == len(y) == len(subjects) == len(sessions)
)

DISCOVERED GDF FILES
A01E.gdf -> subject: A01 | session: S1
A01T.gdf -> subject: A01 | session: S0
A02E.gdf -> subject: A02 | session: S1
A02T.gdf -> subject: A02 | session: S0
A03E.gdf -> subject: A03 | session: S1
A03T.gdf -> subject: A03 | session: S0
A04E.gdf -> subject: A04 | session: S1
A04T.gdf -> subject: A04 | session: S0
A05E.gdf -> subject: A05 | session: S1
A05T.gdf -> subject: A05 | session: S0
A06E.gdf -> subject: A06 | session: S1
A06T.gdf -> subject: A06 | session: S0
A07E.gdf -> subject: A07 | session: S1
A07T.gdf -> subject: A07 | session: S0
A08E.gdf -> subject: A08 | session: S1
A08T.gdf -> subject: A08 | session: S0
A09E.gdf -> subject: A09 | session: S1
A09T.gdf -> subject: A09 | session: S0
A01T.gdf: X=(288, 22, 1000), classes=[72 72 72 72], subject=A01, session=['S0']
A02T.gdf: X=(288, 22, 1000), classes=[72 72 72 72], subject=A02, session=['S0']
A03T.gdf: X=(288, 22, 1000), classes=[72 72 72 72], subject=A03, session=['S0']
A04T.gdf: X=(288, 22, 1000), classes=

In [5]:

# CELL 4 — PREPROCESSING + LEAKAGE-SAFE ALIGNMENT
def apply_main_filter(X, mode, fs):
    if mode == "none":
        return X.astype(np.float32)
    if mode == "1_38":
        sos = signal.butter(5, [1,38], btype="bandpass", fs=fs, output="sos")
    elif mode == "8_30":
        sos = signal.butter(4, [8,30], btype="bandpass", fs=fs, output="sos")
    else:
        raise ValueError(mode)
    return signal.sosfiltfilt(sos, X, axis=-1).astype(np.float32)

class TrainNormalizer:
    def __init__(self, mode="channel"):
        self.mode = mode
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        if self.mode == "global":
            self.mean_ = np.float32(X.mean())
            self.std_ = np.float32(X.std() + 1e-6)
        else:
            self.mean_ = X.mean(axis=(0,2), keepdims=True).astype(np.float32)
            self.std_ = (X.std(axis=(0,2), keepdims=True) + 1e-6).astype(np.float32)
        return self

    def transform(self, X):
        return ((X - self.mean_) / self.std_).astype(np.float32)

def normalized_covariance(trial):
    C = trial @ trial.T
    return C / (np.trace(C) + 1e-8)

def fit_source_whitener(X_source):
    C = np.mean([normalized_covariance(t) for t in X_source], axis=0)
    vals, vecs = np.linalg.eigh(C + 1e-5*np.eye(C.shape[0]))
    W = vecs @ np.diag(1.0/np.sqrt(vals + 1e-5)) @ vecs.T
    return W.astype(np.float32)

def apply_whitener(X, W):
    return np.einsum("ij,njt->nit", W, X).astype(np.float32)

def fit_target_ea_from_calibration(X_cal):
    C = np.mean([normalized_covariance(t) for t in X_cal], axis=0)
    vals, vecs = np.linalg.eigh(C + 1e-5*np.eye(C.shape[0]))
    W = vecs @ np.diag(1.0/np.sqrt(vals + 1e-5)) @ vecs.T
    return W.astype(np.float32)

print("Main filter:", cfg.main_filter)


Main filter: none


In [6]:

# CELL 5 — PAPER-ALIGNED CSP + LASSO
def bandpass_trials(X, lo, hi, fs):
    sos = signal.butter(4, [lo, hi], btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, X, axis=-1).astype(np.float32)

def fit_ovr_csp(X, y, bands, n_comp=4, fs=250):
    all_filters = []
    train_feats = []

    for lo, hi in bands:
        Xb = bandpass_trials(X, lo, hi, fs)
        band_filters = []
        band_feats = []

        for c in range(cfg.n_classes):
            Xc = Xb[y == c]
            Xr = Xb[y != c]
            Rc = np.mean([normalized_covariance(t) for t in Xc], axis=0)
            Rr = np.mean([normalized_covariance(t) for t in Xr], axis=0)
            reg = 1e-6*np.eye(Rc.shape[0])
            vals, vecs = eigh(Rc + reg, Rc + Rr + reg)
            idx = np.argsort(vals)[::-1][:n_comp]
            W = vecs[:, idx]
            band_filters.append(W)

            Z = np.asarray([W.T @ t for t in Xb])
            band_feats.append(np.log(np.var(Z, axis=2) + 1e-8))

        all_filters.append(band_filters)
        train_feats.append(np.concatenate(band_feats, axis=1))

    return np.concatenate(train_feats, axis=1), all_filters

def transform_ovr_csp(X, filters, bands, fs=250):
    outputs = []
    for (lo, hi), band_filters in zip(bands, filters):
        Xb = bandpass_trials(X, lo, hi, fs)
        band_feats = []
        for W in band_filters:
            Z = np.asarray([W.T @ t for t in Xb])
            band_feats.append(np.log(np.var(Z, axis=2) + 1e-8))
        outputs.append(np.concatenate(band_feats, axis=1))
    return np.concatenate(outputs, axis=1)

def fit_csp_lda_source(X_tr, y_tr, X_val, X_te):
    F_tr, filters = fit_ovr_csp(
        X_tr, y_tr, cfg.csp_bands,
        n_comp=cfg.csp_components,
        fs=cfg.sfreq
    )
    F_val = transform_ovr_csp(X_val, filters, cfg.csp_bands, cfg.sfreq)
    F_te = transform_ovr_csp(X_te, filters, cfg.csp_bands, cfg.sfreq)

    scaler = StandardScaler()
    F_tr_s = scaler.fit_transform(F_tr)
    F_val_s = scaler.transform(F_val)
    F_te_s = scaler.transform(F_te)

    if cfg.use_lasso:
        clf_l1 = LogisticRegression(
            penalty="l1", solver="liblinear",
            C=0.2, multi_class="ovr",
            max_iter=2000, random_state=cfg.seed
        )
        clf_l1.fit(F_tr_s, y_tr)
        support = np.any(np.abs(clf_l1.coef_) > 1e-8, axis=0)
        if support.sum() < cfg.n_classes:
            support = np.ones(F_tr_s.shape[1], dtype=bool)
    else:
        support = np.ones(F_tr_s.shape[1], dtype=bool)

    F_tr_s = F_tr_s[:, support]
    F_val_s = F_val_s[:, support]
    F_te_s = F_te_s[:, support]

    lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    lda.fit(F_tr_s, y_tr)

    return {
        "filters": filters,
        "support": support,
        "scaler": scaler,
        "lda": lda,
        "P_tr": lda.predict_proba(F_tr_s),
        "P_val": lda.predict_proba(F_val_s),
        "P_te": lda.predict_proba(F_te_s),
    }

print("CSP bands:", cfg.csp_bands)


CSP bands: ((1, 4), (4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 35), (35, 38))


In [7]:

# CELL 6 — EEGNET + CENTER LOSS
def act(name):
    return {
        "relu": nn.ReLU(inplace=True),
        "elu": nn.ELU(inplace=True),
        "gelu": nn.GELU(),
        "tanh": nn.Tanh()
    }[name.lower()]

class EEGNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        F1, D, F2 = cfg.eegnet_f1, cfg.eegnet_d, cfg.eegnet_f2

        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, (1,64), padding=(0,32), bias=False),
            nn.BatchNorm2d(F1)
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1*D, (cfg.n_channels,1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1,8)),
            nn.Dropout(cfg.dropout)
        )
        self.separable = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1,15), padding=(0,7),
                      groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, 1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1,8)),
            nn.Dropout(cfg.dropout)
        )
        self.feature_dim = cfg.fusion_dim
        self.proj = nn.Linear(F2, cfg.fusion_dim)
        self.classifier = nn.Linear(cfg.fusion_dim, cfg.n_classes)

    def forward(self, x):
        z = self.temporal(x)
        z = self.spatial(z)
        z = self.separable(z)
        z = z.mean(dim=(2,3))
        feat = self.proj(z)
        logits = self.classifier(feat)
        return logits, feat

class CenterLoss(nn.Module):
    def __init__(self, n_classes, feat_dim, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.register_buffer("centers", torch.zeros(n_classes, feat_dim))

    def forward(self, feats, labels):
        centers = self.centers[labels]
        return ((feats-centers)**2).sum(dim=1).mean()

    @torch.no_grad()
    def update(self, feats, labels):
        for c in labels.unique():
            mask = labels == c
            delta = feats[mask].mean(0) - self.centers[c]
            self.centers[c] += self.alpha * delta

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def make_loader(X, y, batch=64, shuffle=True):
    ds = EEGDataset(X, y)
    if not shuffle:
        return DataLoader(ds, batch_size=batch, shuffle=False)
    counts = np.bincount(y, minlength=cfg.n_classes).astype(float)
    weights = 1.0/np.maximum(counts,1)
    sampler = WeightedRandomSampler(
        torch.tensor(weights[y], dtype=torch.double),
        len(y), replacement=True
    )
    return DataLoader(ds, batch_size=batch, sampler=sampler)

def predict_logits(model, X):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), 128):
            xb = torch.tensor(X[i:i+128], dtype=torch.float32, device=DEVICE).unsqueeze(1)
            outs.append(model(xb)[0].cpu().numpy())
    return np.concatenate(outs)

def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p/(p.sum(axis=1, keepdims=True)+1e-12)

def tune_temperature(logits, y_val, grid):
    best_T, best_nll = 1.0, np.inf
    for T in grid:
        p = softmax_np(logits / T)
        nll = log_loss(y_val, p, labels=list(range(cfg.n_classes)))
        if nll < best_nll:
            best_nll, best_T = nll, float(T)
    return best_T, best_nll

def train_eegnet(X_tr, y_tr, X_val, y_val, use_center=True):
    model = EEGNet(cfg).to(DEVICE)
    centers = CenterLoss(cfg.n_classes, cfg.fusion_dim, alpha=0.5).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.epochs, eta_min=1e-6
    )
    ce = nn.CrossEntropyLoss()

    tr_loader = make_loader(X_tr, y_tr, cfg.batch_size, True)
    val_loader = make_loader(X_val, y_val, cfg.batch_size, False)

    best_state, best_val = None, -np.inf
    history, wait = [], 0

    for ep in range(cfg.epochs):
        model.train()
        y_true, y_pred = [], []
        total_loss = 0.0

        for xb, yb in tr_loader:
            xb = xb.to(DEVICE).unsqueeze(1)
            yb = yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            logits, feats = model(xb)
            loss = ce(logits, yb)

            if use_center and ep >= cfg.center_warmup:
                lc = centers(feats, yb)
                loss = loss + cfg.center_lambda*lc
                centers.update(feats.detach(), yb)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            total_loss += float(loss.detach().cpu()) * len(yb)
            y_true.extend(yb.cpu().numpy())
            y_pred.extend(logits.argmax(1).detach().cpu().numpy())

        scheduler.step()

        model.eval()
        val_logits = predict_logits(model, X_val)
        val_acc = accuracy_score(y_val, val_logits.argmax(1))

        row = {
            "epoch": ep,
            "train_loss": total_loss/max(len(y_tr),1),
            "train_acc": accuracy_score(y_true, y_pred),
            "val_acc": val_acc
        }
        history.append(row)

        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= cfg.patience:
            break
        if cfg.fast_dev_run and ep >= 2:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


In [8]:

# CELL 7 — MULTI-SCALE RAW EEG + TRANSFORMER + OPTIONAL BILSTM
class SE(nn.Module):
    def __init__(self, dim, reduction=8):
        super().__init__()
        h = max(dim//reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(dim,h), nn.GELU(), nn.Linear(h,dim), nn.Sigmoid()
        )
    def forward(self, x):
        w = self.mlp(x.mean(1)).unsqueeze(1)
        return x*w

class PatchTokenizer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.proj = nn.Conv2d(
            1, cfg.transformer_dim,
            kernel_size=(cfg.patch_spatial, cfg.patch_temporal),
            stride=(cfg.patch_stride_spatial, cfg.patch_stride_temporal)
        )

    def forward(self, x):
        z = self.proj(x)
        return z.flatten(2).transpose(1,2)

class MultiScaleCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        C = cfg.cnn_dim
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(1,C,(1,k),padding=(0,k//2),bias=False),
                nn.BatchNorm2d(C),
                act(cfg.activation),
                nn.Conv2d(C,C,(cfg.n_channels,1),groups=C,bias=False),
                nn.BatchNorm2d(C),
                act(cfg.activation)
            ) for k in (3,5,7,11,15)
        ])
        self.merge = nn.Sequential(
            nn.Conv2d(C*5,C*2,1,bias=False),
            nn.BatchNorm2d(C*2),
            act(cfg.activation)
        )
        self.attn = SE(C*2)
        self.pool = nn.AvgPool2d((1,8))
        self.proj = nn.Linear(C*2, cfg.fusion_dim)

    def forward(self, x):
        z = torch.cat([b(x) for b in self.branches], dim=1)
        z = self.merge(z)
        tokens = z.mean(2).transpose(1,2)
        pooled = self.attn(tokens).mean(1)
        vec = self.proj(pooled)
        return tokens, vec

class RelationalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=cfg.transformer_dim,
            nhead=cfg.transformer_heads,
            dim_feedforward=cfg.transformer_ff,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.enc = nn.TransformerEncoder(layer, cfg.transformer_layers)
        self.norm = nn.LayerNorm(cfg.transformer_dim)
        self.proj = nn.Linear(cfg.transformer_dim, cfg.fusion_dim)

    def forward(self, tokens):
        z = self.norm(self.enc(tokens))
        return z, self.proj(z.mean(1))

class BiLSTMEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.rnn = nn.LSTM(
            cfg.transformer_dim, cfg.bilstm_hidden,
            num_layers=cfg.bilstm_layers,
            batch_first=True, bidirectional=True,
            dropout=0.2 if cfg.bilstm_layers>1 else 0
        )
        self.proj = nn.Linear(cfg.bilstm_hidden*2, cfg.fusion_dim)
    def forward(self, tokens):
        z,_ = self.rnn(tokens)
        return z, self.proj(z.mean(1))

class HybridBackbone(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tokenizer = PatchTokenizer(cfg)
        self.ms = MultiScaleCNN(cfg)
        self.rel = RelationalEncoder(cfg)
        self.bilstm = BiLSTMEncoder(cfg) if cfg.use_bilstm else None
        self.eegnet = EEGNet(cfg) if cfg.use_eegnet else None

        n = 2 + int(cfg.use_eegnet) + int(cfg.use_bilstm)
        self.gate = nn.Sequential(
            nn.Linear(n*cfg.fusion_dim, cfg.fusion_dim),
            nn.GELU(),
            nn.Linear(cfg.fusion_dim,n)
        )
        self.head = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim),
            nn.LayerNorm(cfg.fusion_dim),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(cfg.fusion_dim,cfg.n_classes)
        )

    def forward(self, x):
        feats = []

        raw_tokens = self.tokenizer(x)
        if raw_tokens.size(1) > 200:
            idx = torch.linspace(0,raw_tokens.size(1)-1,200,device=x.device).long()
            raw_tokens = raw_tokens[:,idx]

        _, raw_rel = self.rel(raw_tokens)
        feats.append(raw_rel)

        ms_tokens, ms_vec = self.ms(x)
        feats.append(ms_vec)

        if self.eegnet is not None:
            eeg_logits, eeg_vec = self.eegnet(x)
            feats.append(eeg_vec)
        else:
            eeg_logits = None

        if self.bilstm is not None:
            _, rnn_vec = self.bilstm(raw_tokens)
            feats.append(rnn_vec)

        cat = torch.cat(feats,dim=1)
        weights = torch.softmax(self.gate(cat),dim=1)
        stack = torch.stack(feats,dim=1)
        fused = (stack*weights.unsqueeze(-1)).sum(1)

        return self.head(fused), {
            "embedding": fused,
            "fusion_weights": weights,
            "eegnet_logits": eeg_logits
        }

def train_hybrid(X_tr,y_tr,X_val,y_val):
    model = HybridBackbone(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt,cfg.epochs,eta_min=1e-6)
    ce = nn.CrossEntropyLoss()

    tr_loader=make_loader(X_tr,y_tr,cfg.batch_size,True)
    best_state,best_val=None,-np.inf
    wait=0

    for ep in range(cfg.epochs):
        model.train()
        for xb,yb in tr_loader:
            xb=xb.to(DEVICE).unsqueeze(1)
            yb=yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits,aux=model(xb)
            loss=ce(logits,yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_logits=[]
            for i in range(0,len(X_val),128):
                xb=torch.tensor(X_val[i:i+128],dtype=torch.float32,device=DEVICE).unsqueeze(1)
                val_logits.append(model(xb)[0].cpu().numpy())
        val_logits=np.concatenate(val_logits)
        val_acc=accuracy_score(y_val,val_logits.argmax(1))

        if val_acc>best_val:
            best_val=val_acc
            best_state=copy.deepcopy(model.state_dict())
            wait=0
        else:
            wait+=1

        if wait>=cfg.patience or (cfg.fast_dev_run and ep>=2):
            break

    model.load_state_dict(best_state)
    return model

def hybrid_probs(model,X):
    model.eval()
    out=[]
    with torch.no_grad():
        for i in range(0,len(X),128):
            xb=torch.tensor(X[i:i+128],dtype=torch.float32,device=DEVICE).unsqueeze(1)
            out.append(model(xb)[0].cpu().numpy())
    return softmax_np(np.concatenate(out))


In [9]:

# CELL 8 — CALIBRATED FUSION
def weighted_fusion(P1, P2, alpha):
    c1=P1.max(1,keepdims=True)
    c2=P2.max(1,keepdims=True)
    w1=alpha*c1
    w2=(1-alpha)*c2
    return (w1*P1+w2*P2)/(w1+w2+1e-12)

def search_alpha(P1,P2,y):
    best=(0.5,-np.inf)
    for a in np.arange(0,1.001,0.05):
        p=weighted_fusion(P1,P2,a)
        acc=accuracy_score(y,p.argmax(1))
        if acc>best[1]:
            best=(float(a),float(acc))
    return best

def fit_stacker(P1,P2,y):
    Xmeta=np.concatenate([P1,P2],axis=1)
    clf=LogisticRegression(
        max_iter=2000,C=2.0,
        multi_class="multinomial",
        class_weight="balanced"
    )
    clf.fit(Xmeta,y)
    return clf, accuracy_score(y,clf.predict(Xmeta))

def evaluate_probs(y_true,P):
    pred=P.argmax(1)
    return {
        "accuracy": accuracy_score(y_true,pred),
        "balanced_accuracy": balanced_accuracy_score(y_true,pred),
        "macro_f1": f1_score(y_true,pred,average="macro"),
        "kappa": cohen_kappa_score(y_true,pred)
    }


In [10]:

# CELL 9 — INNER SOURCE VALIDATION + STRICT LOSO FOLD BUILDER
def split_outer(subjects,target):
    subjects=np.asarray(subjects).astype(str)
    test=np.flatnonzero(subjects==target)
    train=np.flatnonzero(subjects!=target)
    return train,test

def split_inner_subjects(subjects, train_idx, n_val, seed):
    rng=np.random.RandomState(seed)
    src=np.unique(subjects[train_idx])
    if len(src)<n_val+1:
        raise ValueError(
            f"Need at least {n_val+1} source subjects, found {len(src)}."
        )
    val_subj=rng.choice(src,size=n_val,replace=False)
    mask=np.isin(subjects[train_idx],val_subj)
    return train_idx[~mask], train_idx[mask], val_subj.tolist()

def preprocess_source_fold(X_tr,X_val,X_te):
    X_tr=apply_main_filter(X_tr,cfg.main_filter,cfg.sfreq)
    X_val=apply_main_filter(X_val,cfg.main_filter,cfg.sfreq)
    X_te=apply_main_filter(X_te,cfg.main_filter,cfg.sfreq)

    normalizer=TrainNormalizer(cfg.normalize_mode).fit(X_tr)
    X_tr=normalizer.transform(X_tr)
    X_val=normalizer.transform(X_val)
    X_te=normalizer.transform(X_te)

    W=None
    if cfg.use_source_whitening:
        W=fit_source_whitener(X_tr)
        X_tr=apply_whitener(X_tr,W)
        X_val=apply_whitener(X_val,W)
        X_te=apply_whitener(X_te,W)

    return X_tr,X_val,X_te,normalizer,W

def target_adaptation_split(test_idx,subjects,sessions,S0,S1):
    m_cal=(sessions[test_idx]==S0)
    m_test=(sessions[test_idx]==S1)
    cal_idx=test_idx[m_cal]
    te_idx=test_idx[m_test]
    if len(cal_idx)==0 or len(te_idx)==0:
        raise RuntimeError("Target adaptation requires both calibration and test sessions.")
    return cal_idx,te_idx

print("Inner subject validation is source-only.")


Inner subject validation is source-only.


In [18]:
# ============================================================
# CELL — STRICT LOSO RUNNER
# ============================================================

def run_strict_loso(X, y, subjects):

    X = np.asarray(X)
    y = np.asarray(y)
    subjects = np.asarray(subjects).astype(str)

    # --------------------------------------------------------
    # GLOBAL VALIDATION
    # --------------------------------------------------------

    assert len(X) == len(y)
    assert len(X) == len(subjects)

    outer_subjects = sorted(
        np.unique(subjects).tolist()
    )

    print("\n" + "=" * 80)
    print("STRICT LOSO")
    print("=" * 80)

    print(
        "Outer subjects:",
        outer_subjects
    )

    print(
        "Number of subjects:",
        len(outer_subjects)
    )

    # --------------------------------------------------------
    # MUST BE 9 SUBJECTS
    # --------------------------------------------------------

    if len(outer_subjects) != 9:

        raise RuntimeError(
            "\nLOSO cannot start because subject metadata "
            "is incorrect.\n\n"
            f"Detected subjects:\n{outer_subjects}\n\n"
            "Expected:\n"
            "A01 ... A09"
        )

    results = []
    fold_store = {}

    # ========================================================
    # OUTER LOSO
    # ========================================================

    for fold, target in enumerate(
        outer_subjects,
        start=1
    ):

        print("\n" + "=" * 80)
        print(
            f"STRICT LOSO {fold}/9 — TARGET {target}"
        )
        print("=" * 80)

        # ----------------------------------------------------
        # OUTER SPLIT
        # ----------------------------------------------------

        outer_train_idx = np.flatnonzero(
            subjects != target
        )

        outer_test_idx = np.flatnonzero(
            subjects == target
        )

        source_subjects = sorted(
            np.unique(
                subjects[outer_train_idx]
            ).tolist()
        )

        test_subjects = sorted(
            np.unique(
                subjects[outer_test_idx]
            ).tolist()
        )

        print(
            "\nSource subjects:",
            source_subjects
        )

        print(
            "Test subject:",
            test_subjects
        )

        print(
            "Training trials:",
            len(outer_train_idx)
        )

        print(
            "Test trials:",
            len(outer_test_idx)
        )

        # ----------------------------------------------------
        # LEAKAGE CHECK
        # ----------------------------------------------------

        assert target not in source_subjects

        assert test_subjects == [target]

        assert len(
            set(source_subjects)
            &
            set(test_subjects)
        ) == 0

        # ----------------------------------------------------
        # INNER VALIDATION
        # ----------------------------------------------------

        inner_train_idx, val_idx, val_subjects = (
            split_inner_subjects(
                subjects,
                outer_train_idx,
                cfg.n_inner_val_subjects,
                cfg.seed
            )
        )

        print(
            "\nInner training subjects:",
            sorted(
                np.unique(
                    subjects[inner_train_idx]
                ).tolist()
            )
        )

        print(
            "Validation subjects:",
            sorted(
                np.unique(
                    subjects[val_idx]
                ).tolist()
            )
        )

        # ----------------------------------------------------
        # PREPROCESS
        # ----------------------------------------------------

        Xtr, Xval, Xtest, normalizer, W = (
            preprocess_source_fold(
                X[inner_train_idx],
                X[val_idx],
                X[outer_test_idx]
            )
        )

        ytr = y[inner_train_idx]
        yval = y[val_idx]
        ytest = y[outer_test_idx]

        # ----------------------------------------------------
        # EXPERT 1 — CSP/LDA
        # ----------------------------------------------------

        print("\nTraining CSP-LDA...")

        csp = fit_csp_lda_source(
            Xtr,
            ytr,
            Xval,
            Xtest
        )

        # ----------------------------------------------------
        # EXPERT 2 — EEGNET
        # ----------------------------------------------------

        print("Training EEGNet + CenterLoss...")

        eeg_model, eeg_hist = train_eegnet(
            Xtr,
            ytr,
            Xval,
            yval,
            use_center=True
        )

        logits_eeg_val = predict_logits(
            eeg_model,
            Xval
        )

        logits_eeg_test = predict_logits(
            eeg_model,
            Xtest
        )

        T_eeg, _ = tune_temperature(
            logits_eeg_val,
            yval,
            cfg.temperature_grid
        )

        P_eeg_val = softmax_np(
            logits_eeg_val / T_eeg
        )

        P_eeg_test = softmax_np(
            logits_eeg_test / T_eeg
        )

        # ----------------------------------------------------
        # EXPERT 3 — HYBRID
        # ----------------------------------------------------

        print(
            "Training NF-EEG + MultiScale + Transformer..."
        )

        hybrid = train_hybrid(
            Xtr,
            ytr,
            Xval,
            yval
        )

        P_h_val = hybrid_probs(
            hybrid,
            Xval
        )

        P_h_test = hybrid_probs(
            hybrid,
            Xtest
        )

        # ----------------------------------------------------
        # TEMPERATURE FOR HYBRID
        # ----------------------------------------------------

        T_h, _ = tune_temperature(
            np.log(P_h_val + 1e-12),
            yval,
            cfg.temperature_grid
        )

        P_h_val = softmax_np(
            np.log(P_h_val + 1e-12) / T_h
        )

        P_h_test = softmax_np(
            np.log(P_h_test + 1e-12) / T_h
        )

        # ----------------------------------------------------
        # FUSION 1
        # CSP + EEGNet
        # ----------------------------------------------------

        alpha_eeg, _ = search_alpha(
            csp["P_val"],
            P_eeg_val,
            yval
        )

        P_csp_eeg_val = weighted_fusion(
            csp["P_val"],
            P_eeg_val,
            alpha_eeg
        )

        P_csp_eeg_test = weighted_fusion(
            csp["P_te"],
            P_eeg_test,
            alpha_eeg
        )

        # ----------------------------------------------------
        # FUSION 2
        # CSP + HYBRID
        # ----------------------------------------------------

        alpha_hybrid, _ = search_alpha(
            csp["P_val"],
            P_h_val,
            yval
        )

        P_csp_h_val = weighted_fusion(
            csp["P_val"],
            P_h_val,
            alpha_hybrid
        )

        P_csp_h_test = weighted_fusion(
            csp["P_te"],
            P_h_test,
            alpha_hybrid
        )

        # ----------------------------------------------------
        # VALIDATION SCORES
        # ----------------------------------------------------

        candidates = {

            "CSP_LDA":
                (
                    evaluate_probs(
                        yval,
                        csp["P_val"]
                    )["balanced_accuracy"],

                    csp["P_te"]
                ),

            "EEGNet":
                (
                    evaluate_probs(
                        yval,
                        P_eeg_val
                    )["balanced_accuracy"],

                    P_eeg_test
                ),

            "Hybrid":
                (
                    evaluate_probs(
                        yval,
                        P_h_val
                    )["balanced_accuracy"],

                    P_h_test
                ),

            "CSP_EEGNet":
                (
                    evaluate_probs(
                        yval,
                        P_csp_eeg_val
                    )["balanced_accuracy"],

                    P_csp_eeg_test
                ),

            "CSP_Hybrid":
                (
                    evaluate_probs(
                        yval,
                        P_csp_h_val
                    )["balanced_accuracy"],

                    P_csp_h_test
                )
        }

        # ----------------------------------------------------
        # MODEL SELECTION
        # ----------------------------------------------------

        best_method = max(
            candidates,
            key=lambda name:
                candidates[name][0]
        )

        best_validation_score, P_best = (
            candidates[best_method]
        )

        # ----------------------------------------------------
        # OUTER TEST — USED ONLY NOW
        # ----------------------------------------------------

        test_metrics = evaluate_probs(
            ytest,
            P_best
        )

        print(
            "\nSelected model:",
            best_method
        )

        print(
            "Inner validation balanced accuracy:",
            f"{best_validation_score*100:.2f}%"
        )

        print(
            "OUTER TEST ACCURACY:",
            f"{test_metrics['accuracy']*100:.2f}%"
        )

        print(
            "OUTER TEST BALANCED ACCURACY:",
            f"{test_metrics['balanced_accuracy']*100:.2f}%"
        )

        print(
            "OUTER TEST MACRO F1:",
            f"{test_metrics['macro_f1']*100:.2f}%"
        )

        print(
            "OUTER TEST KAPPA:",
            f"{test_metrics['kappa']:.4f}"
        )

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        results.append({
            "fold": fold,
            "subject": target,

            "accuracy":
                test_metrics["accuracy"],

            "balanced_accuracy":
                test_metrics[
                    "balanced_accuracy"
                ],

            "macro_f1":
                test_metrics["macro_f1"],

            "kappa":
                test_metrics["kappa"],

            "best_method":
                best_method,

            "validation_subjects":
                ",".join(
                    sorted(
                        np.unique(
                            subjects[val_idx]
                        ).tolist()
                    )
                ),

            "temperature_eegnet":
                T_eeg,

            "temperature_hybrid":
                T_h,

            "alpha_csp_eegnet":
                alpha_eeg,

            "alpha_csp_hybrid":
                alpha_hybrid
        })

        fold_store[target] = {
            "y_test": ytest,
            "P_best": P_best,
            "best_method": best_method,
            "normalizer": normalizer,
            "source_whitening": W,
            "csp": csp,
            "eeg_history": eeg_hist
        }

        # ----------------------------------------------------
        # CLEANUP
        # ----------------------------------------------------

        del eeg_model
        del hybrid

        gc.collect()

        if DEVICE.type == "mps":
            torch.mps.empty_cache()

    # ========================================================
    # FINAL LOSO RESULTS
    # ========================================================

    results_df = pd.DataFrame(
        results
    )

    print("\n" + "=" * 80)
    print("STRICT LOSO COMPLETE")
    print("=" * 80)

    print(
        results_df[
            [
                "subject",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "kappa",
                "best_method"
            ]
        ].to_string(index=False)
    )

    mean_acc = results_df[
        "accuracy"
    ].mean()

    std_acc = results_df[
        "accuracy"
    ].std(ddof=1)

    ci = stats.t.interval(
        0.95,
        len(results_df) - 1,
        loc=mean_acc,
        scale=stats.sem(
            results_df["accuracy"]
        )
    )

    print("\nFINAL LOSO ACCURACY")
    print(
        f"Mean : {mean_acc*100:.2f}%"
    )
    print(
        f"Std  : {std_acc*100:.2f}%"
    )
    print(
        f"95% CI: "
        f"{ci[0]*100:.2f}% – "
        f"{ci[1]*100:.2f}%"
    )

    return (
        results_df,
        fold_store
    )

In [19]:

# CELL 11 — TARGET-ADAPTATION LOSO
class SimpleGenerator(nn.Module):
    def __init__(self, noise_dim=1600):
        super().__init__()
        self.fc=nn.Linear(noise_dim,128*16*125)
        self.net=nn.Sequential(
            nn.ConvTranspose2d(128,128,(3,15),(1,3),(1,6)),
            nn.BatchNorm2d(128),nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d(128,128,(3,15),(1,3),(1,6)),
            nn.BatchNorm2d(128),nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d(128,64,(3,5),(1,2),(1,2)),
            nn.BatchNorm2d(64),nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d(64,32,(4,5),(2,1),(1,2)),
            nn.BatchNorm2d(32),nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d(32,1,(1,2),(1,1),(0,0))
        )
    def forward(self,z):
        x=self.fc(z).view(z.size(0),128,16,125)
        x=self.net(x)
        return torch.tanh(F.interpolate(x,size=(22,1000),mode="bilinear",align_corners=False)).squeeze(1)

class DiscPhi(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,(1,23),padding=(0,11)),
            nn.LeakyReLU(0.2,True),
            nn.Conv2d(16,32,(22,1)),
            nn.LeakyReLU(0.2,True),
            nn.Conv2d(32,32,(1,17),padding=(0,8)),
            nn.LeakyReLU(0.2,True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.net(x.unsqueeze(1)).squeeze(-1)

class DiscPsi(nn.Module):
    def __init__(self, sparse_dim):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(sparse_dim,128),
            nn.LeakyReLU(0.2,True),
            nn.Linear(128,64),
            nn.LeakyReLU(0.2,True),
            nn.Linear(64,1)
        )
    def forward(self,x):
        return self.net(x).squeeze(-1)

def build_sparse_transform(X_cal,y_cal):
    F_cal,filters=fit_ovr_csp(
        X_cal,y_cal,cfg.csp_bands,cfg.csp_components,cfg.sfreq
    )
    support=np.ones(F_cal.shape[1],dtype=bool)
    if cfg.use_lasso:
        scaler=StandardScaler().fit(F_cal)
        clf=LogisticRegression(
            penalty="l1",solver="liblinear",C=0.2,
            multi_class="ovr",max_iter=2000
        ).fit(scaler.transform(F_cal),y_cal)
        support=np.any(np.abs(clf.coef_)>1e-8,axis=0)

    def transform(X):
        F=transform_ovr_csp(
            X,filters,cfg.csp_bands,cfg.sfreq
        )
        return F[:,support].astype(np.float32)

    return transform,F_cal[:,support]

def train_one_class_gan(X_cls,sparse_fn,sparse_dim):
    lo=float(X_cls.min())
    hi=float(X_cls.max())
    scale=max(abs(lo),abs(hi),1e-6)

    real=np.clip(X_cls/scale,-1,1).astype(np.float32)
    loader=DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(real)),
        batch_size=min(cfg.gan_batch,len(real)),
        shuffle=True,drop_last=len(real)>=cfg.gan_batch
    )

    G=SimpleGenerator(cfg.gan_noise_dim).to(DEVICE)
    D1=DiscPhi().to(DEVICE)
    D2=DiscPsi(sparse_dim).to(DEVICE)

    optG=torch.optim.Adam(G.parameters(),lr=cfg.gan_lr_g,betas=(0.5,0.999))
    optD=torch.optim.Adam(
        list(D1.parameters())+list(D2.parameters()),
        lr=cfg.gan_lr_d,betas=(0.5,0.999)
    )
    bce=nn.BCEWithLogitsLoss()

    for ep in range(cfg.gan_epochs if not cfg.fast_dev_run else 3):
        for (rb,) in loader:
            rb=rb.to(DEVICE)
            bs=len(rb)

            z=torch.randn(bs,cfg.gan_noise_dim,device=DEVICE)
            fb=G(z)

            optD.zero_grad(set_to_none=True)
            lossD=bce(D1(rb),torch.full((bs,),0.9,device=DEVICE))
            lossD+=bce(D1(fb.detach()),torch.full((bs,),0.1,device=DEVICE))

            real_sp=torch.tensor(
                sparse_fn((rb.detach().cpu().numpy()*scale).astype(np.float32)),
                dtype=torch.float32,device=DEVICE
            )
            fake_sp=torch.tensor(
                sparse_fn((fb.detach().cpu().numpy()*scale).astype(np.float32)),
                dtype=torch.float32,device=DEVICE
            )
            lossD+=bce(D2(real_sp),torch.ones(bs,device=DEVICE))
            lossD+=bce(D2(fake_sp.detach()),torch.zeros(bs,device=DEVICE))
            lossD.backward()
            optD.step()

            optG.zero_grad(set_to_none=True)
            z=torch.randn(bs,cfg.gan_noise_dim,device=DEVICE)
            fb=G(z)
            fake_sp=torch.tensor(
                sparse_fn((fb.detach().cpu().numpy()*scale).astype(np.float32)),
                dtype=torch.float32,device=DEVICE
            )
            lossG=bce(D1(fb),torch.ones(bs,device=DEVICE))
            lossG+=bce(D2(fake_sp),torch.ones(bs,device=DEVICE))
            lossG.backward()
            optG.step()

    return G,scale

def sample_gan(G,scale,n):
    G.eval()
    out=[]
    with torch.no_grad():
        for i in range(0,n,32):
            z=torch.randn(min(32,n-i),cfg.gan_noise_dim,device=DEVICE)
            out.append((G(z).cpu().numpy()*scale).astype(np.float32))
    return np.concatenate(out)

def qc_summary(real,fake):
    def cov(t):
        return normalized_covariance(t)
    c1=np.mean([cov(t) for t in real],axis=0)
    c2=np.mean([cov(t) for t in fake],axis=0)
    f1,p1=signal.welch(real.mean(1),fs=cfg.sfreq,nperseg=256,axis=-1)
    f2,p2=signal.welch(fake.mean(1),fs=cfg.sfreq,nperseg=256,axis=-1)
    return {
        "mean_abs_mean_diff":float(abs(real.mean()-fake.mean())),
        "std_ratio":float(fake.std()/(real.std()+1e-8)),
        "psd_log_distance":float(np.mean(abs(np.log(p1+1e-8)-np.log(p2+1e-8)))),
        "cov_distance":float(np.linalg.norm(c1-c2)/(np.linalg.norm(c1)+1e-8))
    }

def conservative_target_finetune(model,X_cal,y_cal):
    for p in model.parameters():
        p.requires_grad=True

    for name,p in model.named_parameters():
        if "temporal" in name:
            p.requires_grad=False

    opt=torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.target_ft_lr,weight_decay=cfg.weight_decay
    )
    loader=make_loader(X_cal,y_cal,min(32,len(y_cal)),True)

    best=copy.deepcopy(model.state_dict())
    best_acc=-np.inf

    for ep in range(cfg.target_ft_epochs if not cfg.fast_dev_run else 2):
        model.train()
        for xb,yb in loader:
            xb=xb.to(DEVICE).unsqueeze(1)
            yb=yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits,_=model(xb)
            loss=F.cross_entropy(logits,yb)
            loss.backward()
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                0.5
            )
            opt.step()

        acc=accuracy_score(
            y_cal,
            predict_logits(model,X_cal).argmax(1)
        )

        if acc>best_acc:
            best_acc=acc
            best=copy.deepcopy(model.state_dict())

    model.load_state_dict(best)
    return model

def run_target_adaptation_loso(X,y,subjects,sessions,S0="S0",S1="S1"):
    rows=[]
    stores={}

    for fold,target in enumerate(sorted(np.unique(subjects)),1):
        print(f"\nTARGET-ADAPTATION {fold}/9 — {target}")

        outer_tr,outer_te=split_outer(subjects,target)
        inner_tr,val_idx,val_subj=split_inner_subjects(
            subjects,outer_tr,cfg.n_inner_val_subjects,cfg.seed
        )
        cal_idx,test_idx=target_adaptation_split(
            outer_te,subjects,sessions,S0,S1
        )

        # Source-only preprocessing fit.
        Xtr,Xval,_,norm,W=preprocess_source_fold(
            X[inner_tr],X[val_idx],X[test_idx]
        )

        # Target calibration/test preprocessing uses only train-derived
        # normalization. Optional EA is fitted on S0 only.
        Xcal=apply_main_filter(X[cal_idx],cfg.main_filter,cfg.sfreq)
        Xtest=apply_main_filter(X[test_idx],cfg.main_filter,cfg.sfreq)
        Xcal=norm.transform(Xcal)
        Xtest=norm.transform(Xtest)

        if cfg.target_use_ea:
            target_W=fit_target_ea_from_calibration(Xcal)
            Xcal=apply_whitener(Xcal,target_W)
            Xtest=apply_whitener(Xtest,target_W)

        # Source-pretrained neural expert.
        model,_=train_eegnet(
            Xtr,y[inner_tr],Xval,y[val_idx],use_center=True
        )

        # Target calibration fine-tuning.
        model=conservative_target_finetune(
            model,Xcal,y[cal_idx]
        )

        P_cal=softmax_np(predict_logits(model,Xcal))
        P_test=softmax_np(predict_logits(model,Xtest))

        # Temperature is now allowed because this is target-adaptation.
        T,_=tune_temperature(
            np.log(P_cal+1e-12),y[cal_idx],cfg.temperature_grid
        )
        P_cal=softmax_np(np.log(P_cal+1e-12)/T)
        P_test=softmax_np(np.log(P_test+1e-12)/T)

        # CSP/LDA from target calibration is also allowed in this protocol.
        F_cal,filters=fit_ovr_csp(
            Xcal,y[cal_idx],cfg.csp_bands,cfg.csp_components,cfg.sfreq
        )
        F_test=transform_ovr_csp(
            Xtest,filters,cfg.csp_bands,cfg.sfreq
        )
        scaler=StandardScaler().fit(F_cal)
        clf=LinearDiscriminantAnalysis(
            solver="lsqr",shrinkage="auto"
        ).fit(scaler.transform(F_cal),y[cal_idx])
        P_csp_cal=clf.predict_proba(scaler.transform(F_cal))
        P_csp_test=clf.predict_proba(scaler.transform(F_test))

        a,_=search_alpha(P_csp_cal,P_cal,y[cal_idx])
        P_fused=weighted_fusion(P_csp_test,P_test,a)

        # Optional class-wise FBGAN.
        fake_info={}
        if cfg.use_fbgan:
            sparse_fn,sparse_train=build_sparse_transform(
                Xcal,y[cal_idx]
            )
            fake_X=[]; fake_y=[]
            for c in range(cfg.n_classes):
                Xc=Xcal[y[cal_idx]==c]
                G,scale=train_one_class_gan(
                    Xc,sparse_fn,sparse_train.shape[1]
                )
                Xfc=sample_gan(
                    G,scale,
                    cfg.gan_fake_per_class
                )
                fake_X.append(Xfc)
                fake_y.append(np.full(len(Xfc),c,np.int64))
                fake_info[c]=qc_summary(
                    Xc,Xfc[:len(Xc)]
                )

        metrics=evaluate_probs(
            y[test_idx],P_fused
        )

        rows.append({
            "fold":fold,
            "subject":target,
            **metrics,
            "calibration_trials":len(cal_idx),
            "temperature":T,
            "alpha_csp_eegnet":a,
            "used_target_ea":cfg.target_use_ea,
            "used_fbgan":cfg.use_fbgan
        })

        stores[target]={
            "P_test":P_fused,
            "y_test":y[test_idx],
            "qc":fake_info
        }

    return pd.DataFrame(rows),stores


In [20]:

# CELL 12 — FINAL EXPERIMENT DRIVER
print("""
RECOMMENDED EXPERIMENT ORDER
----------------------------
1. Strict LOSO:
   - CSP-LDA
   - EEGNet+CenterLoss
   - NF/MultiScale+Transformer
   - calibrated fusion

2. Only after strict results are frozen:
   Target-adaptation LOSO:
   - S0 calibration
   - optional S0 EA
   - S0 fine-tuning
   - optional FBGAN
   - S0-calibrated fusion
   - S1 final test

3. Report them separately.
""")

# Strict experiment:
strict_results, strict_store = run_strict_loso(X,y,subjects)
display(strict_results)

#Target-adaptation experiment:
#sessions should contain exactly the two session labels used below.
ta_results, ta_store = run_target_adaptation_loso(
    X,y,subjects,sessions,
    S0=np.unique(sessions)[0],
    S1=np.unique(sessions)[1]
)
display(ta_results)



RECOMMENDED EXPERIMENT ORDER
----------------------------
1. Strict LOSO:
   - CSP-LDA
   - EEGNet+CenterLoss
   - NF/MultiScale+Transformer
   - calibrated fusion

2. Only after strict results are frozen:
   Target-adaptation LOSO:
   - S0 calibration
   - optional S0 EA
   - S0 fine-tuning
   - optional FBGAN
   - S0-calibrated fusion
   - S1 final test

3. Report them separately.


STRICT LOSO
Outer subjects: ['A']
Number of subjects: 1


RuntimeError: 
LOSO cannot start because subject metadata is incorrect.

Detected subjects:
['A']

Expected:
A01 ... A09

In [22]:
# ============================================================
# CELL — DEFINITIVE BNCI IV-2a SUBJECT + SESSION METADATA
# ============================================================

import re
from pathlib import Path
import numpy as np


def extract_subject_id(filename):
    name = Path(filename).name.upper()

    match = re.search(r"A\d{2}", name)

    if match is None:
        raise ValueError(
            f"Could not extract subject ID from: {filename}"
        )

    return match.group(0)


def extract_session_id(filename):
    """
    BNCI IV-2a:

        A01T.gdf -> S0 / training session
        A01E.gdf -> S1 / evaluation session
    """

    name = Path(filename).name.upper()

    if re.search(r"A\d{2}T\.GDF$", name):
        return "S0"

    if re.search(r"A\d{2}E\.GDF$", name):
        return "S1"

    raise ValueError(
        f"Could not determine T/E session from: {filename}"
    )


# ============================================================
# DISCOVER FILES
# ============================================================

gdf_files = sorted(
    str(p)
    for p in Path(cfg.data_root).rglob("*.gdf")
)


print("=" * 80)
print("GDF FILE ORDER")
print("=" * 80)

for i, path in enumerate(gdf_files):

    print(
        f"{i:02d}",
        Path(path).name,
        "-> subject:",
        extract_subject_id(path),
        "| session:",
        extract_session_id(path)
    )


# ============================================================
# EXPECTED FILE STRUCTURE
# ============================================================

expected_files = [
    f"A{s:02d}{session}.gdf"
    for s in range(1, 10)
    for session in ["E", "T"]
]

detected_files = [
    Path(p).name
    for p in gdf_files
]

print("\nDetected files:")
print(detected_files)


# Don't require alphabetical order to match our generated list.
missing_files = sorted(
    set(expected_files) - set(detected_files)
)

extra_files = sorted(
    set(detected_files) - set(expected_files)
)

if missing_files:

    raise RuntimeError(
        "Missing expected BNCI IV-2a files:\n"
        + "\n".join(missing_files)
    )

if extra_files:

    print(
        "\nWARNING: Extra GDF files detected:"
    )

    for f in extra_files:
        print(" ", f)


# ============================================================
# REBUILD METADATA FROM ACTUAL FILES
# ============================================================

# IMPORTANT:
# X was built in the same loop over gdf_files.
#
# Therefore each file contributes exactly the number of trials
# contained in that file.
#
# We need to reconstruct subject/session labels using the
# actual trial counts instead of assuming 288 unconditionally.

if len(X) != len(y):

    raise RuntimeError(
        f"X/y mismatch: X={len(X)}, y={len(y)}"
    )


# ------------------------------------------------------------
# We need the file-level trial counts.
#
# Safest approach: reload only metadata/event counts.
# This avoids depending on the stale `subjects` variable.
# ------------------------------------------------------------

file_trial_counts = []

for path in gdf_files:

    raw = mne.io.read_raw_gdf(
        path,
        preload=False,
        verbose=False
    )

    events, event_id = mne.events_from_annotations(
        raw,
        verbose=False
    )

    reverse = {}

    for name, code in event_id.items():

        nums = re.findall(
            r"\d+",
            str(name)
        )

        if nums:
            reverse[code] = int(
                nums[-1]
            )

    valid_count = 0

    for ev in events:

        onset, _, internal_code = ev

        true_code = reverse.get(
            internal_code,
            internal_code
        )

        if true_code not in EVENT_MAP:
            continue

        start = int(onset)
        stop = start + cfg.n_times

        if stop <= raw.n_times:

            valid_count += 1

    file_trial_counts.append(
        valid_count
    )

    print(
        f"{Path(path).name}: "
        f"{valid_count} valid trials"
    )


# ============================================================
# CHECK TOTAL
# ============================================================

total_from_files = sum(
    file_trial_counts
)

print("\nTrials according to GDF files:")
print(
    "Total:",
    total_from_files
)

print(
    "X contains:",
    len(X)
)

if total_from_files != len(X):

    raise RuntimeError(
        "\nFile-level trial count does not match X.\n"
        f"GDF total = {total_from_files}\n"
        f"X total   = {len(X)}\n\n"
        "This indicates that X was not built from the "
        "same file list/order currently being inspected."
    )


# ============================================================
# BUILD ONE SUBJECT + SESSION LABEL PER TRIAL
# ============================================================

subject_blocks = []
session_blocks = []

for path, n_trials in zip(
    gdf_files,
    file_trial_counts
):

    sid = extract_subject_id(
        path
    )

    session = extract_session_id(
        path
    )

    subject_blocks.append(
        np.full(
            n_trials,
            sid,
            dtype=str
        )
    )

    session_blocks.append(
        np.full(
            n_trials,
            session,
            dtype=str
        )
    )


subjects = np.concatenate(
    subject_blocks
)

sessions = np.concatenate(
    session_blocks
)


# ============================================================
# FINAL VALIDATION
# ============================================================

assert len(subjects) == len(X)

assert len(sessions) == len(X)

assert len(y) == len(X)


unique_subjects = sorted(
    np.unique(subjects).tolist()
)

unique_sessions = sorted(
    np.unique(sessions).tolist()
)


print("\n" + "=" * 80)
print("FINAL DATASET METADATA")
print("=" * 80)

print(
    "X shape       :",
    X.shape
)

print(
    "y shape       :",
    y.shape
)

print(
    "subjects shape:",
    subjects.shape
)

print(
    "sessions shape:",
    sessions.shape
)

print(
    "\nSubjects:",
    unique_subjects
)

print(
    "Sessions:",
    unique_sessions
)


# ============================================================
# SUBJECT DISTRIBUTION
# ============================================================

print("\nSubject distribution:")

for sid, count in zip(
    *np.unique(
        subjects,
        return_counts=True
    )
):

    print(
        f"  {sid}: {count}"
    )


# ============================================================
# SUBJECT × SESSION DISTRIBUTION
# ============================================================

print("\nSubject × Session:")

for sid in unique_subjects:

    for session in ["S0", "S1"]:

        count = int(
            np.sum(
                (subjects == sid) &
                (sessions == session)
            )
        )

        print(
            f"  {sid} {session}: {count}"
        )


# ============================================================
# FINAL SAFETY CHECKS
# ============================================================

assert unique_subjects == [
    "A01", "A02", "A03",
    "A04", "A05", "A06",
    "A07", "A08", "A09"
]

assert unique_sessions == [
    "S0", "S1"
]

assert len(subjects) == 5184

assert len(sessions) == 5184

print("\n" + "=" * 80)
print("SUBJECT / SESSION METADATA READY")
print("=" * 80)

print(
    "Strict LOSO can now use all 9 subjects."
)

print(
    "Target adaptation can use S0 calibration -> S1 test."
)

GDF FILE ORDER
00 A01E.gdf -> subject: A01 | session: S1
01 A01T.gdf -> subject: A01 | session: S0
02 A02E.gdf -> subject: A02 | session: S1
03 A02T.gdf -> subject: A02 | session: S0
04 A03E.gdf -> subject: A03 | session: S1
05 A03T.gdf -> subject: A03 | session: S0
06 A04E.gdf -> subject: A04 | session: S1
07 A04T.gdf -> subject: A04 | session: S0
08 A05E.gdf -> subject: A05 | session: S1
09 A05T.gdf -> subject: A05 | session: S0
10 A06E.gdf -> subject: A06 | session: S1
11 A06T.gdf -> subject: A06 | session: S0
12 A07E.gdf -> subject: A07 | session: S1
13 A07T.gdf -> subject: A07 | session: S0
14 A08E.gdf -> subject: A08 | session: S1
15 A08T.gdf -> subject: A08 | session: S0
16 A09E.gdf -> subject: A09 | session: S1
17 A09T.gdf -> subject: A09 | session: S0

Detected files:
['A01E.gdf', 'A01T.gdf', 'A02E.gdf', 'A02T.gdf', 'A03E.gdf', 'A03T.gdf', 'A04E.gdf', 'A04T.gdf', 'A05E.gdf', 'A05T.gdf', 'A06E.gdf', 'A06T.gdf', 'A07E.gdf', 'A07T.gdf', 'A08E.gdf', 'A08T.gdf', 'A09E.gdf', 'A09T.g

AssertionError: 

In [14]:

# CELL 13 — FINAL REPORTING / STATISTICS
def summarize_subject_results(df):
    acc=df["accuracy"].values
    mean=acc.mean()
    std=acc.std(ddof=1)
    ci=stats.t.interval(
        0.95,len(acc)-1,
        loc=mean,
        scale=stats.sem(acc)
    )
    return pd.DataFrame([{
        "mean_accuracy_pct":100*mean,
        "std_pct":100*std,
        "ci95_low_pct":100*ci[0],
        "ci95_high_pct":100*ci[1],
        "median_pct":100*np.median(acc),
        "min_subject_pct":100*acc.min(),
        "max_subject_pct":100*acc.max(),
        "mean_balanced_accuracy_pct":100*df["balanced_accuracy"].mean(),
        "mean_macro_f1_pct":100*df["macro_f1"].mean(),
        "mean_kappa":df["kappa"].mean()
    }])

def paired_subject_test(df_a,df_b):
    m=df_a[["subject","accuracy"]].merge(
        df_b[["subject","accuracy"]],
        on="subject",suffixes=("_a","_b")
    )
    d=m.accuracy_a.values-m.accuracy_b.values
    t=stats.ttest_rel(
        m.accuracy_a,
        m.accuracy_b
    )
    dz=d.mean()/(d.std(ddof=1)+1e-12)
    return {
        "mean_difference":float(d.mean()),
        "paired_t":float(t.statistic),
        "p_value":float(t.pvalue),
        "cohen_dz":float(dz)
    }

def plot_subject_results(df,title):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9,4))
    plt.bar(df["subject"],df["accuracy"]*100)
    plt.axhline(25,linestyle="--",linewidth=1)
    plt.ylabel("Accuracy (%)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Example after execution:
# print(summarize_subject_results(strict_results))
# plot_subject_results(strict_results,"Strict LOSO")


In [24]:
# ============================================================
# MASTER CELL — CONVORELENET + TRANSFER LEARNING
# ============================================================
#
# Reproduces the architecture and transfer-learning strategy
# described in:
#
# Otarbay & Kyzyrkanov (2026)
# "Transfer learning for subject-independent motor imagery
#  EEG classification using convolutional relational networks"
#
# IMPLEMENTED:
#   - 22-channel raw EEG
#   - 4-class MI
#   - strict subject-wise LOSO
#   - 8-30 Hz zero-phase Butterworth filtering
#   - training-only normalization
#   - two parallel CNN branches
#   - CNN feature fusion -> 128 dimensions
#   - positional encoding
#   - 8-layer Transformer
#   - 3-layer BiLSTM
#   - temporal mean pooling
#   - MLP classifier
#   - source-subject pretraining
#   - conservative target fine-tuning
#   - held-out target test evaluation
#   - accuracy / macro-F1 / kappa
#   - per-subject results
#   - confusion matrices
#
# INPUT:
#   eeg_bundle.pkl
#
# EXPECTED:
#   bundle["X_raw"]    -> (N, 22, 1000)
#   bundle["y"]        -> integer labels 0..3
#   bundle["subjects"] -> subject IDs
#   bundle["sessions"] -> e.g. "0train", "1test"
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import copy
import time
import pickle
import random
import warnings

import numpy as np
import scipy.signal as sig
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

# ----------------------------
# Reproducibility
# ----------------------------
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ----------------------------
# Device
# ----------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("=" * 75)
print("CONVORELENET TRANSFER-LEARNING MASTER PIPELINE")
print("=" * 75)
print("Device:", DEVICE)


# ============================================================
# 2. PAPER-INSPIRED HYPERPARAMETERS
# ============================================================

# Input
FS = 250

# Paper uses 8-30 Hz preprocessing
FILTER_LOW = 8.0
FILTER_HIGH = 30.0
FILTER_ORDER = 4


# ----------------------------
# CNN
# ----------------------------

DEEP_FILTERS = 64
SHALLOW_FILTERS = 80

FUSED_DIM = 128

CNN_DROPOUT = 0.40


# ----------------------------
# Transformer
# ----------------------------

TRANSFORMER_LAYERS = 8
TRANSFORMER_HEADS = 4
TRANSFORMER_DIM = 128

TRANSFORMER_DROPOUT = 0.10

TRANSFORMER_FF_DIM = 256


# ----------------------------
# BiLSTM
# ----------------------------

LSTM_LAYERS = 3
LSTM_HIDDEN = 128
LSTM_DROPOUT = 0.30


# ----------------------------
# Classification head
# ----------------------------

HEAD_HIDDEN = 128
HEAD_DROPOUT = 0.50


# ============================================================
# 3. TRAINING SETTINGS
# ============================================================

# Source pretraining
PRETRAIN_EPOCHS = 150
PRETRAIN_LR = 1e-3
PRETRAIN_BATCH = 64


# Conservative target fine-tuning
# Paper's conservative strategy uses lower LR / fewer updates.
FT_EPOCHS = 75
FT_LR = 5e-5
FT_BATCH = 32


WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0


# Fine-tuning policy
# The paper describes convolutional and recurrent layers as
# unfrozen while Transformer layers are partially frozen.
#
# Here we provide a conservative implementation:
#   CNN            -> trainable
#   fusion         -> trainable
#   transformer    -> frozen by default
#   LSTM           -> trainable
#   classifier     -> trainable
#
FREEZE_TRANSFORMER = True


# ============================================================
# 4. DATA SETTINGS
# ============================================================

BUNDLE_PATH = "eeg_bundle.pkl"

# Session names expected from your current bundle
CAL_SESSION_DEFAULT = "0train"
TEST_SESSION_DEFAULT = "1test"


# Set to None to automatically use all subjects
MAX_SUBJECTS = None


# Set True for full LOSO
RUN_FULL_LOSO = True


# Save results
RESULTS_PATH = "convo_relenet_tl_loso_results.pkl"


# ============================================================
# 5. LOAD DATA
# ============================================================

print("\n[1/12] Loading EEG bundle...")

if not os.path.exists(BUNDLE_PATH):
    raise FileNotFoundError(
        f"Could not find {BUNDLE_PATH}. "
        f"Place eeg_bundle.pkl in the current working directory."
    )

with open(BUNDLE_PATH, "rb") as f:
    bundle = pickle.load(f)


X_raw = np.asarray(bundle["X_raw"], dtype=np.float32)
y = np.asarray(bundle["y"], dtype=np.int64)
subjects = np.asarray(bundle["subjects"]).astype(str)
sessions = np.asarray(bundle["sessions"]).astype(str)

FS_BUNDLE = int(bundle.get("FS", FS))
CAL_SESSION = str(bundle.get("S0", CAL_SESSION_DEFAULT))
TEST_SESSION = str(bundle.get("S1", TEST_SESSION_DEFAULT))

CLASSES = bundle.get("classes", None)

if CLASSES is None:
    CLASSES = sorted(np.unique(y).tolist())

CLASSES = [str(c) for c in CLASSES]

N_CLASSES = len(np.unique(y))
N_CHANNELS = X_raw.shape[1]
N_TIME = X_raw.shape[2]


print("X shape       :", X_raw.shape)
print("y shape       :", y.shape)
print("Subjects      :", np.unique(subjects))
print("Classes       :", np.unique(y))
print("Class names   :", CLASSES)
print("Sampling rate :", FS_BUNDLE)
print("Calibration   :", CAL_SESSION)
print("Test session  :", TEST_SESSION)


if N_CHANNELS != 22:
    print(
        "\nWARNING:"
        f" Expected 22 channels for IV-2a, got {N_CHANNELS}."
    )

if N_CLASSES != 4:
    print(
        "\nWARNING:"
        f" Expected 4 classes for IV-2a, got {N_CLASSES}."
    )


# ============================================================
# 6. SUBJECT ORDER
# ============================================================

def subject_sort_key(s):
    s = str(s)

    digits = "".join(ch for ch in s if ch.isdigit())

    if digits:
        return (0, int(digits))

    return (1, s)


ALL_SUBJECTS = sorted(
    np.unique(subjects).tolist(),
    key=subject_sort_key
)


if MAX_SUBJECTS is not None:
    ALL_SUBJECTS = ALL_SUBJECTS[:MAX_SUBJECTS]


print("\nSubjects used:")
print(ALL_SUBJECTS)


# ============================================================
# 7. PREPROCESSING
# ============================================================

def bandpass_zero_phase(
    X,
    low=FILTER_LOW,
    high=FILTER_HIGH,
    fs=FS,
    order=FILTER_ORDER
):
    """
    Zero-phase Butterworth band-pass filter.

    X:
        (N, C, T)
    """

    nyquist = fs / 2.0

    if high >= nyquist:
        raise ValueError(
            f"High cutoff {high} must be below Nyquist {nyquist}."
        )

    b, a = sig.butter(
        order,
        [
            low / nyquist,
            high / nyquist
        ],
        btype="band"
    )

    Xf = sig.filtfilt(
        b,
        a,
        X,
        axis=-1
    )

    return Xf.astype(np.float32)


def fit_standardizer(X):
    """
    Fit standardization using ONLY training data.

    Per-channel standardization across trials/time.
    """

    mean = X.mean(axis=(0, 2), keepdims=True)

    std = X.std(axis=(0, 2), keepdims=True)

    std = np.maximum(std, 1e-6)

    return mean.astype(np.float32), std.astype(np.float32)


def apply_standardizer(X, mean, std):
    return (
        (X - mean) / std
    ).astype(np.float32)


def preprocess_fold(
    X_train,
    X_cal,
    X_test
):

    print("    Filtering 8-30 Hz...")

    X_train = bandpass_zero_phase(
        X_train,
        fs=FS_BUNDLE
    )

    X_cal = bandpass_zero_phase(
        X_cal,
        fs=FS_BUNDLE
    )

    X_test = bandpass_zero_phase(
        X_test,
        fs=FS_BUNDLE
    )

    print("    Fitting training-only normalization...")

    mean, std = fit_standardizer(X_train)

    X_train = apply_standardizer(
        X_train,
        mean,
        std
    )

    X_cal = apply_standardizer(
        X_cal,
        mean,
        std
    )

    X_test = apply_standardizer(
        X_test,
        mean,
        std
    )

    return X_train, X_cal, X_test, mean, std


# ============================================================
# 8. POSITIONAL ENCODING
# ============================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=4096
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            *
            (
                -np.log(10000.0)
                /
                d_model
            )
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        # x = B,T,D

        T = x.size(1)

        return (
            x
            +
            self.pe[:, :T]
        )


# ============================================================
# 9. CONVORELENET MODEL
# ============================================================

class ConvoReleNet(nn.Module):

    def __init__(
        self,
        n_channels,
        n_time,
        n_classes,
        activation="elu"
    ):

        super().__init__()

        self.n_channels = n_channels
        self.n_time = n_time
        self.n_classes = n_classes


        # ====================================================
        # Activation
        # ====================================================

        activation = activation.lower()

        if activation == "elu":

            def act():
                return nn.ELU(inplace=True)

        elif activation == "relu":

            def act():
                return nn.ReLU(inplace=True)

        elif activation == "tanh":

            def act():
                return nn.Tanh()

        else:
            raise ValueError(
                f"Unknown activation: {activation}"
            )


        # ====================================================
        # DEEP CNN BRANCH
        # ====================================================
        #
        # The deep branch learns temporal-spectral
        # representations.
        #
        # Input:
        #     B,1,C,T
        #
        # ====================================================

        self.deep_branch = nn.Sequential(

            # temporal convolution
            nn.Conv2d(
                1,
                16,
                kernel_size=(1, 3),
                padding=(0, 1),
                bias=False
            ),

            nn.BatchNorm2d(16),

            act(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),

            nn.Dropout(
                CNN_DROPOUT
            ),


            nn.Conv2d(
                16,
                32,
                kernel_size=(1, 5),
                padding=(0, 2),
                bias=False
            ),

            nn.BatchNorm2d(32),

            act(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),

            nn.Dropout(
                CNN_DROPOUT
            ),


            nn.Conv2d(
                32,
                DEEP_FILTERS,
                kernel_size=(1, 7),
                padding=(0, 3),
                bias=False
            ),

            nn.BatchNorm2d(
                DEEP_FILTERS
            ),

            act(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),

            nn.Dropout(
                CNN_DROPOUT
            )
        )


        # ====================================================
        # SHALLOW CNN BRANCH
        # ====================================================
        #
        # Spatial + temporal representation.
        #
        # EEG:
        #     B,1,C,T
        #
        # Spatial convolution across channels.
        #
        # ====================================================

        self.shallow_branch = nn.Sequential(

            nn.Conv2d(
                1,
                40,
                kernel_size=(1, 7),
                padding=(0, 3),
                bias=False
            ),

            nn.BatchNorm2d(40),

            act(),


            nn.Conv2d(
                40,
                SHALLOW_FILTERS,
                kernel_size=(n_channels, 1),
                bias=False
            ),

            nn.BatchNorm2d(
                SHALLOW_FILTERS
            ),

            act(),

            nn.AvgPool2d(
                kernel_size=(1, 4)
            ),

            nn.Dropout(
                CNN_DROPOUT
            )
        )


        # ====================================================
        # Determine CNN output temporal dimensions
        # ====================================================

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_time
            )

            deep_dummy = self.deep_branch(
                dummy
            )

            shallow_dummy = self.shallow_branch(
                dummy
            )

        if deep_dummy.shape[-1] != shallow_dummy.shape[-1]:

            # Align both temporal resolutions
            self.align_temporal = True

        else:

            self.align_temporal = False


        # ====================================================
        # Feature projection
        # ====================================================

        self.feature_fusion = nn.Sequential(

            nn.Conv1d(
                DEEP_FILTERS + SHALLOW_FILTERS,
                FUSED_DIM,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm1d(
                FUSED_DIM
            ),

            act()
        )


        # ====================================================
        # Positional encoding
        # ====================================================

        self.position = PositionalEncoding(
            FUSED_DIM,
            max_len=4096
        )


        # ====================================================
        # Transformer
        # ====================================================

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=TRANSFORMER_DIM,
            nhead=TRANSFORMER_HEADS,
            dim_feedforward=TRANSFORMER_FF_DIM,
            dropout=TRANSFORMER_DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=False
        )


        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=TRANSFORMER_LAYERS
        )


        # ====================================================
        # BiLSTM
        # ====================================================

        self.bilstm = nn.LSTM(
            input_size=TRANSFORMER_DIM,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=(
                LSTM_DROPOUT
                if LSTM_LAYERS > 1
                else 0.0
            )
        )


        # ====================================================
        # Classification head
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Linear(
                LSTM_HIDDEN * 2,
                HEAD_HIDDEN
            ),

            act(),

            nn.Dropout(
                HEAD_DROPOUT
            ),

            nn.Linear(
                HEAD_HIDDEN,
                n_classes
            )
        )


    def forward(
        self,
        x,
        return_features=False
    ):

        # Input:
        # B,C,T

        x = x.unsqueeze(1)

        # B,1,C,T


        # ================================================
        # Deep branch
        # ================================================

        deep = self.deep_branch(
            x
        )


        # ================================================
        # Shallow branch
        # ================================================

        shallow = self.shallow_branch(
            x
        )


        # ================================================
        # Align temporal dimensions
        # ================================================

        target_t = min(
            deep.shape[-1],
            shallow.shape[-1]
        )

        if deep.shape[-1] != target_t:

            deep = F.adaptive_avg_pool2d(
                deep,
                (
                    deep.shape[-2],
                    target_t
                )
            )

        if shallow.shape[-1] != target_t:

            shallow = F.adaptive_avg_pool2d(
                shallow,
                (
                    shallow.shape[-2],
                    target_t
                )
            )


        # ================================================
        # Remove singleton spatial dimension
        # ================================================

        deep = deep.mean(
            dim=2
        )

        shallow = shallow.mean(
            dim=2
        )

        # B,F,T


        # ================================================
        # Concatenate branches
        # ================================================

        fused = torch.cat(
            [
                deep,
                shallow
            ],
            dim=1
        )


        # ================================================
        # Project to 128
        # ================================================

        fused = self.feature_fusion(
            fused
        )

        # B,128,T


        # ================================================
        # Transformer sequence
        # ================================================

        fused = fused.transpose(
            1,
            2
        )

        # B,T,128


        fused = self.position(
            fused
        )


        # ================================================
        # Transformer
        # ================================================

        trans = self.transformer(
            fused
        )


        # ================================================
        # BiLSTM
        # ================================================

        lstm_out, _ = self.bilstm(
            trans
        )


        # ================================================
        # Temporal mean pooling
        # ================================================

        pooled = lstm_out.mean(
            dim=1
        )


        # ================================================
        # Classification
        # ================================================

        logits = self.classifier(
            pooled
        )


        if return_features:

            return logits, pooled

        return logits


# ============================================================
# 10. MODEL PARAMETER COUNT
# ============================================================

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def count_all_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
    )


# ============================================================
# 11. DATA LOADERS
# ============================================================

def make_loader(
    X,
    y,
    batch_size,
    shuffle=True,
    drop_last=False
):

    dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(X, dtype=np.float32)
        ),
        torch.from_numpy(
            np.asarray(y, dtype=np.int64)
        )
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last
    )


# ============================================================
# 12. EVALUATION
# ============================================================

def evaluate_model(
    model,
    X,
    y,
    batch_size=128
):

    model.eval()

    loader = make_loader(
        X,
        y,
        batch_size=batch_size,
        shuffle=False
    )

    all_logits = []
    all_true = []

    with torch.no_grad():

        for xb, yb in loader:

            xb = xb.to(DEVICE)

            logits = model(
                xb
            )

            all_logits.append(
                logits.cpu().numpy()
            )

            all_true.append(
                yb.numpy()
            )


    logits = np.concatenate(
        all_logits
    )

    true = np.concatenate(
        all_true
    )

    # stable softmax
    z = logits - logits.max(
        axis=1,
        keepdims=True
    )

    probs = np.exp(z)

    probs /= (
        probs.sum(
            axis=1,
            keepdims=True
        )
        +
        1e-12
    )

    pred = probs.argmax(
        axis=1
    )


    acc = accuracy_score(
        true,
        pred
    ) * 100.0


    f1 = f1_score(
        true,
        pred,
        average="macro"
    )


    kappa = cohen_kappa_score(
        true,
        pred
    )


    return {
        "accuracy": acc,
        "macro_f1": f1,
        "kappa": kappa,
        "pred": pred,
        "true": true,
        "probs": probs,
        "logits": logits
    }


# ============================================================
# 13. TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    X,
    y,
    epochs,
    lr,
    batch_size,
    phase_name="training",
    verbose=True
):

    loader = make_loader(
        X,
        y,
        batch_size=batch_size,
        shuffle=True
    )

    optimizer = optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )


    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=max(
            lr * 0.01,
            1e-7
        )
    )


    criterion = nn.CrossEntropyLoss()


    history = {
        "epoch": [],
        "loss": [],
        "accuracy": [],
        "lr": []
    }


    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_acc = -np.inf


    for epoch in range(1, epochs + 1):

        model.train()

        total_loss = 0.0
        total_correct = 0
        total_n = 0


        for xb, yb in loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)


            optimizer.zero_grad(
                set_to_none=True
            )


            logits = model(
                xb
            )


            loss = criterion(
                logits,
                yb
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            optimizer.step()


            total_loss += (
                loss.item()
                *
                xb.size(0)
            )


            total_correct += (
                logits.argmax(
                    dim=1
                )
                ==
                yb
            ).sum().item()


            total_n += xb.size(0)


        scheduler.step()


        epoch_loss = (
            total_loss
            /
            max(total_n, 1)
        )

        epoch_acc = (
            100.0
            *
            total_correct
            /
            max(total_n, 1)
        )


        current_lr = optimizer.param_groups[0]["lr"]


        history["epoch"].append(
            epoch
        )

        history["loss"].append(
            epoch_loss
        )

        history["accuracy"].append(
            epoch_acc
        )

        history["lr"].append(
            current_lr
        )


        if epoch_acc > best_acc:

            best_acc = epoch_acc

            best_state = copy.deepcopy(
                model.state_dict()
            )


        if verbose:

            if (
                epoch == 1
                or
                epoch % 10 == 0
                or
                epoch == epochs
            ):

                print(
                    f"      {phase_name:<18s} "
                    f"epoch {epoch:3d}/{epochs} | "
                    f"loss={epoch_loss:.4f} | "
                    f"acc={epoch_acc:6.2f}% | "
                    f"lr={current_lr:.2e}"
                )


    model.load_state_dict(
        best_state
    )

    return model, history


# ============================================================
# 14. FREEZE / UNFREEZE FOR TRANSFER LEARNING
# ============================================================

def configure_transfer_learning(
    model,
    freeze_transformer=True
):

    # First: everything trainable
    for p in model.parameters():

        p.requires_grad = True


    if freeze_transformer:

        for p in model.transformer.parameters():

            p.requires_grad = False


    # Position encoding has no trainable parameters.


def show_trainable_parameters(model):

    trainable = count_parameters(
        model
    )

    total = count_all_parameters(
        model
    )

    print(
        f"Trainable parameters: "
        f"{trainable:,}"
    )

    print(
        f"Total parameters    : "
        f"{total:,}"
    )


# ============================================================
# 15. BUILD LOSO FOLD
# ============================================================

def prepare_loso_data(
    target_subject
):

    print(
        f"\nPreparing target subject "
        f"{target_subject}"
    )


    # --------------------------------------------------------
    # Source subjects
    # --------------------------------------------------------

    source_mask = (
        subjects
        !=
        target_subject
    )


    target_mask = (
        subjects
        ==
        target_subject
    )


    X_source = X_raw[
        source_mask
    ]

    y_source = y[
        source_mask
    ]


    X_target = X_raw[
        target_mask
    ]

    y_target = y[
        target_mask
    ]

    target_sessions = sessions[
        target_mask
    ]


    # --------------------------------------------------------
    # Target calibration
    # --------------------------------------------------------

    cal_mask = (
        target_sessions
        ==
        CAL_SESSION
    )


    test_mask = (
        target_sessions
        ==
        TEST_SESSION
    )


    X_cal = X_target[
        cal_mask
    ]

    y_cal = y_target[
        cal_mask
    ]


    X_test = X_target[
        test_mask
    ]

    y_test = y_target[
        test_mask
    ]


    if len(X_cal) == 0:

        raise ValueError(
            f"Target subject {target_subject} "
            f"has no calibration session "
            f"{CAL_SESSION!r}."
        )


    if len(X_test) == 0:

        raise ValueError(
            f"Target subject {target_subject} "
            f"has no test session "
            f"{TEST_SESSION!r}."
        )


    # --------------------------------------------------------
    # Preprocessing
    # --------------------------------------------------------

    X_source, X_cal, X_test, mean, std = preprocess_fold(
        X_source,
        X_cal,
        X_test
    )


    print(
        "    Source:",
        X_source.shape
    )

    print(
        "    Calibration:",
        X_cal.shape
    )

    print(
        "    Test:",
        X_test.shape
    )


    return {
        "X_source": X_source,
        "y_source": y_source,

        "X_cal": X_cal,
        "y_cal": y_cal,

        "X_test": X_test,
        "y_test": y_test,

        "norm_mean": mean,
        "norm_std": std
    }


# ============================================================
# 16. SINGLE LOSO FOLD
# ============================================================

def run_single_fold(
    target_subject,
    verbose=True
):

    start_time = time.time()


    print("\n")
    print("=" * 75)
    print(
        f"TARGET SUBJECT: {target_subject}"
    )
    print("=" * 75)


    data = prepare_loso_data(
        target_subject
    )


    X_source = data["X_source"]
    y_source = data["y_source"]

    X_cal = data["X_cal"]
    y_cal = data["y_cal"]

    X_test = data["X_test"]
    y_test = data["y_test"]


    # ========================================================
    # SOURCE MODEL
    # ========================================================

    print(
        "\n[2/12] Creating source model..."
    )


    source_model = ConvoReleNet(
        n_channels=X_source.shape[1],
        n_time=X_source.shape[2],
        n_classes=N_CLASSES,
        activation="elu"
    ).to(DEVICE)


    print(
        "Source model:"
    )

    show_trainable_parameters(
        source_model
    )


    # ========================================================
    # SOURCE PRETRAINING
    # ========================================================

    print(
        "\n[3/12] Source-subject pretraining..."
    )


    source_model, pretrain_history = train_model(
        source_model,
        X_source,
        y_source,
        epochs=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR,
        batch_size=PRETRAIN_BATCH,
        phase_name="pretrain",
        verbose=verbose
    )


    # ========================================================
    # SOURCE MODEL BEFORE TARGET ADAPTATION
    # ========================================================

    print(
        "\n[4/12] Evaluating pretrained source model "
        "on target calibration/test..."
    )


    source_cal_metrics = evaluate_model(
        source_model,
        X_cal,
        y_cal
    )


    source_test_metrics = evaluate_model(
        source_model,
        X_test,
        y_test
    )


    print(
        f"    Source → target calibration: "
        f"{source_cal_metrics['accuracy']:.2f}%"
    )

    print(
        f"    Source → target test       : "
        f"{source_test_metrics['accuracy']:.2f}%"
    )


    # ========================================================
    # TARGET ADAPTATION
    # ========================================================

    print(
        "\n[5/12] Configuring conservative "
        "transfer learning..."
    )


    target_model = copy.deepcopy(
        source_model
    )


    configure_transfer_learning(
        target_model,
        freeze_transformer=FREEZE_TRANSFORMER
    )


    show_trainable_parameters(
        target_model
    )


    # ========================================================
    # TARGET FINE-TUNING
    # ========================================================

    print(
        "\n[6/12] Conservative target fine-tuning..."
    )


    target_model, ft_history = train_model(
        target_model,
        X_cal,
        y_cal,
        epochs=FT_EPOCHS,
        lr=FT_LR,
        batch_size=FT_BATCH,
        phase_name="target-ft",
        verbose=verbose
    )


    # ========================================================
    # CALIBRATION PERFORMANCE
    # ========================================================

    print(
        "\n[7/12] Evaluating adapted model..."
    )


    cal_metrics = evaluate_model(
        target_model,
        X_cal,
        y_cal
    )


    print(
        f"    Calibration accuracy: "
        f"{cal_metrics['accuracy']:.2f}%"
    )

    print(
        f"    Calibration macro-F1: "
        f"{cal_metrics['macro_f1']:.4f}"
    )

    print(
        f"    Calibration kappa   : "
        f"{cal_metrics['kappa']:.4f}"
    )


    # ========================================================
    # TARGET TEST
    # ========================================================

    print(
        "\n[8/12] Evaluating completely held-out "
        "target test..."
    )


    test_metrics = evaluate_model(
        target_model,
        X_test,
        y_test
    )


    print(
        f"\n    TARGET TEST ACCURACY : "
        f"{test_metrics['accuracy']:.2f}%"
    )

    print(
        f"    TARGET TEST MACRO-F1 : "
        f"{test_metrics['macro_f1']:.4f}"
    )

    print(
        f"    TARGET TEST KAPPA    : "
        f"{test_metrics['kappa']:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_metrics["pred"],
        labels=np.arange(N_CLASSES)
    )


    # Per-class accuracy
    per_class_acc = {}

    for cls in range(N_CLASSES):

        mask = (
            y_test
            ==
            cls
        )

        if mask.sum() == 0:

            per_class_acc[
                cls
            ] = np.nan

        else:

            per_class_acc[
                cls
            ] = (
                test_metrics["pred"][mask]
                ==
                cls
            ).mean() * 100.0


    # ========================================================
    # RESULT OBJECT
    # ========================================================

    elapsed = time.time() - start_time


    result = {

        "subject": str(
            target_subject
        ),

        "n_source": len(X_source),
        "n_cal": len(X_cal),
        "n_test": len(X_test),

        "source_test_accuracy":
            source_test_metrics["accuracy"],

        "source_cal_accuracy":
            source_cal_metrics["accuracy"],

        "cal_accuracy":
            cal_metrics["accuracy"],

        "cal_macro_f1":
            cal_metrics["macro_f1"],

        "cal_kappa":
            cal_metrics["kappa"],

        "test_accuracy":
            test_metrics["accuracy"],

        "test_macro_f1":
            test_metrics["macro_f1"],

        "test_kappa":
            test_metrics["kappa"],

        "test_true":
            test_metrics["true"],

        "test_pred":
            test_metrics["pred"],

        "test_probs":
            test_metrics["probs"],

        "confusion_matrix":
            cm,

        "per_class_accuracy":
            per_class_acc,

        "pretrain_history":
            pretrain_history,

        "finetune_history":
            ft_history,

        "model":
            target_model.state_dict(),

        "parameter_count":
            count_all_parameters(
                target_model
            ),

        "elapsed_seconds":
            elapsed
    }


    print(
        "\nFold completed in "
        f"{elapsed / 60.0:.2f} minutes."
    )


    return result


# ============================================================
# 17. RUN LOSO
# ============================================================

all_results = {}


if RUN_FULL_LOSO:

    print("\n")
    print("=" * 75)
    print("STARTING FULL LOSO")
    print("=" * 75)


    for fold_idx, target_subject in enumerate(
        ALL_SUBJECTS,
        start=1
    ):

        print(
            f"\nFold {fold_idx}/"
            f"{len(ALL_SUBJECTS)}"
        )

        try:

            result = run_single_fold(
                target_subject,
                verbose=True
            )

            all_results[
                str(target_subject)
            ] = result


        except Exception as e:

            print(
                "\nERROR in subject "
                f"{target_subject}:"
            )

            print(
                repr(e)
            )

            all_results[
                str(target_subject)
            ] = {
                "subject":
                    str(target_subject),
                "error":
                    repr(e)
            }


else:

    # Run first subject only
    target_subject = ALL_SUBJECTS[0]

    all_results[
        target_subject
    ] = run_single_fold(
        target_subject,
        verbose=True
    )


# ============================================================
# 18. SUMMARY TABLE
# ============================================================

print("\n")
print("=" * 75)
print("LOSO SUMMARY")
print("=" * 75)


summary_rows = []


for subject, result in all_results.items():

    if "error" in result:

        summary_rows.append(
            {
                "subject": subject,
                "source_cal": np.nan,
                "source_test": np.nan,
                "cal_accuracy": np.nan,
                "test_accuracy": np.nan,
                "macro_f1": np.nan,
                "kappa": np.nan
            }
        )

        continue


    summary_rows.append(
        {
            "subject":
                subject,

            "source_cal":
                result[
                    "source_cal_accuracy"
                ],

            "source_test":
                result[
                    "source_test_accuracy"
                ],

            "cal_accuracy":
                result[
                    "cal_accuracy"
                ],

            "test_accuracy":
                result[
                    "test_accuracy"
                ],

            "macro_f1":
                result[
                    "test_macro_f1"
                ],

            "kappa":
                result[
                    "test_kappa"
                ]
        }
    )


# Sort
summary_rows = sorted(
    summary_rows,
    key=lambda x:
        subject_sort_key(
            x["subject"]
        )
)


# Print manually so pandas is not required
print(
    f"{'Subject':>10s} "
    f"{'SourceCal':>12s} "
    f"{'SourceTest':>12s} "
    f"{'AdaptCal':>12s} "
    f"{'TestAcc':>12s} "
    f"{'MacroF1':>10s} "
    f"{'Kappa':>10s}"
)

print("-" * 85)


valid_test_acc = []
valid_f1 = []
valid_kappa = []


for row in summary_rows:

    print(
        f"{row['subject']:>10s} "
        f"{row['source_cal']:12.2f} "
        f"{row['source_test']:12.2f} "
        f"{row['cal_accuracy']:12.2f} "
        f"{row['test_accuracy']:12.2f} "
        f"{row['macro_f1']:10.4f} "
        f"{row['kappa']:10.4f}"
    )


    if np.isfinite(
        row["test_accuracy"]
    ):

        valid_test_acc.append(
            row["test_accuracy"]
        )

        valid_f1.append(
            row["macro_f1"]
        )

        valid_kappa.append(
            row["kappa"]
        )


# ============================================================
# 19. FINAL METRICS
# ============================================================

mean_acc = (
    np.mean(valid_test_acc)
    if len(valid_test_acc)
    else np.nan
)

std_acc = (
    np.std(valid_test_acc)
    if len(valid_test_acc)
    else np.nan
)

mean_f1 = (
    np.mean(valid_f1)
    if len(valid_f1)
    else np.nan
)

std_f1 = (
    np.std(valid_f1)
    if len(valid_f1)
    else np.nan
)

mean_kappa = (
    np.mean(valid_kappa)
    if len(valid_kappa)
    else np.nan
)

std_kappa = (
    np.std(valid_kappa)
    if len(valid_kappa)
    else np.nan
)


print("\n")
print("=" * 75)
print("FINAL CONVORELENET + TL RESULT")
print("=" * 75)

print(
    f"Mean accuracy : "
    f"{mean_acc:.2f}%"
)

print(
    f"STD accuracy  : "
    f"{std_acc:.2f}%"
)

print(
    f"Mean Macro-F1 : "
    f"{mean_f1:.4f}"
)

print(
    f"STD Macro-F1  : "
    f"{std_f1:.4f}"
)

print(
    f"Mean Kappa    : "
    f"{mean_kappa:.4f}"
)

print(
    f"STD Kappa     : "
    f"{std_kappa:.4f}"
)

print(
    f"Chance level  : "
    f"{100.0 / N_CLASSES:.2f}%"
)


# ============================================================
# 20. PAPER COMPARISON
# ============================================================

print("\n")
print("=" * 75)
print("REFERENCE VALUES FROM THE PAPER")
print("=" * 75)

print(
    "ConvoReleNet baseline IV-2a : "
    "72.22 ± 20.49%"
)

print(
    "ConvoReleNet + TL IV-2a     : "
    "79.44 ± 11.09%"
)

print(
    "ConvoReleNet + TL + Tanh    : "
    "87.55 ± 9.64%"
)


if np.isfinite(mean_acc):

    print(
        "\nDifference from paper "
        "compact TL result:"
    )

    print(
        f"{mean_acc - 79.44:+.2f} percentage points"
    )

    print(
        "\nDifference from paper "
        "Tanh result:"
    )

    print(
        f"{mean_acc - 87.55:+.2f} percentage points"
    )


# ============================================================
# 21. CONFUSION MATRICES
# ============================================================

valid_subjects = [
    s
    for s, r in all_results.items()
    if "error" not in r
]


if len(valid_subjects) > 0:

    n = len(valid_subjects)

    ncols = 3

    nrows = int(
        np.ceil(
            n / ncols
        )
    )

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(
            5 * ncols,
            4.5 * nrows
        )
    )


    axes = np.array(
        axes
    ).reshape(
        -1
    )


    for i, subject in enumerate(
        valid_subjects
    ):

        result = all_results[
            subject
        ]

        cm = result[
            "confusion_matrix"
        ]


        ax = axes[i]

        im = ax.imshow(
            cm,
            interpolation="nearest",
            cmap="Blues"
        )


        ax.set_title(
            f"Subject {subject}\n"
            f"Accuracy = "
            f"{result['test_accuracy']:.2f}%"
        )


        ax.set_xlabel(
            "Predicted"
        )

        ax.set_ylabel(
            "True"
        )


        ax.set_xticks(
            range(N_CLASSES)
        )

        ax.set_yticks(
            range(N_CLASSES)
        )

        ax.set_xticklabels(
            CLASSES
        )

        ax.set_yticklabels(
            CLASSES
        )


        for r in range(
            N_CLASSES
        ):

            for c in range(
                N_CLASSES
            ):

                ax.text(
                    c,
                    r,
                    str(cm[r, c]),
                    ha="center",
                    va="center"
                )


        fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04
        )


    for i in range(
        len(valid_subjects),
        len(axes)
    ):

        axes[i].axis(
            "off"
        )


    plt.suptitle(
        "ConvoReleNet + Transfer Learning — LOSO Confusion Matrices",
        fontsize=14,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 22. SUBJECT ACCURACY PLOT
# ============================================================

if len(valid_subjects) > 0:

    subjects_plot = [
        s
        for s in valid_subjects
    ]

    acc_plot = [
        all_results[s][
            "test_accuracy"
        ]
        for s in subjects_plot
    ]


    plt.figure(
        figsize=(
            max(
                10,
                len(subjects_plot) * 1.2
            ),
            5
        )
    )


    bars = plt.bar(
        np.arange(
            len(subjects_plot)
        ),
        acc_plot
    )


    plt.axhline(
        25.0,
        linestyle=":",
        linewidth=1.5,
        label="Chance"
    )


    plt.axhline(
        79.44,
        linestyle="--",
        linewidth=1.2,
        label="Paper TL = 79.44%"
    )


    plt.axhline(
        87.55,
        linestyle="--",
        linewidth=1.2,
        label="Paper Tanh = 87.55%"
    )


    plt.xticks(
        np.arange(
            len(subjects_plot)
        ),
        [
            f"A{s}"
            for s in subjects_plot
        ]
    )


    plt.ylabel(
        "Test Accuracy (%)"
    )

    plt.xlabel(
        "Held-out Subject"
    )

    plt.title(
        "ConvoReleNet + Transfer Learning — Subject-wise LOSO Accuracy",
        fontweight="bold"
    )


    plt.ylim(
        0,
        100
    )


    plt.grid(
        axis="y",
        alpha=0.3
    )


    plt.legend()

    plt.tight_layout()

    plt.show()


# ============================================================
# 23. PER-CLASS PERFORMANCE
# ============================================================

print("\n")
print("=" * 75)
print("PER-CLASS ACCURACY")
print("=" * 75)


global_cm = np.zeros(
    (N_CLASSES, N_CLASSES),
    dtype=np.int64
)


for subject in valid_subjects:

    global_cm += (
        all_results[
            subject
        ][
            "confusion_matrix"
        ]
    )


for cls in range(
    N_CLASSES
):

    total = global_cm[
        cls
    ].sum()


    if total == 0:

        acc = np.nan

    else:

        acc = (
            global_cm[
                cls,
                cls
            ]
            /
            total
            *
            100.0
        )


    print(
        f"{CLASSES[cls]:>15s}: "
        f"{acc:.2f}%"
    )


# ============================================================
# 24. SAVE RESULTS
# ============================================================

print("\n")
print(
    "Saving results..."
)


save_payload = {

    "paper":
        "Otarbay & Kyzyrkanov 2026",

    "architecture":
        "ConvoReleNet",

    "protocol":
        "subject-wise LOSO + target calibration fine-tuning",

    "mean_accuracy":
        mean_acc,

    "std_accuracy":
        std_acc,

    "mean_macro_f1":
        mean_f1,

    "std_macro_f1":
        std_f1,

    "mean_kappa":
        mean_kappa,

    "std_kappa":
        std_kappa,

    "classes":
        CLASSES,

    "subjects":
        ALL_SUBJECTS,

    "results":
        all_results
}


with open(
    RESULTS_PATH,
    "wb"
) as f:

    pickle.dump(
        save_payload,
        f
    )


print(
    f"Saved: {RESULTS_PATH}"
)


# ============================================================
# 25. FINAL STATUS
# ============================================================

print("\n")
print("=" * 75)
print("MASTER CELL COMPLETE")
print("=" * 75)

print(
    "Architecture:"
)

print(
    "  Dual CNN branches"
)

print(
    "  -> 128-d fusion"
)

print(
    "  -> positional encoding"
)

print(
    f"  -> Transformer x {TRANSFORMER_LAYERS}"
)

print(
    f"  -> BiLSTM x {LSTM_LAYERS}"
)

print(
    "  -> temporal mean pooling"
)

print(
    "  -> MLP classifier"
)

print(
    "\nTransfer protocol:"
)

print(
    "  Source subjects -> pretraining"
)

print(
    "  Held-out subject calibration -> conservative fine-tuning"
)

print(
    "  Held-out subject test -> final evaluation"
)

print(
    "\nFinal performance:"
)

print(
    f"  Accuracy = {mean_acc:.2f} ± {std_acc:.2f}%"
)

print(
    f"  Macro-F1 = {mean_f1:.4f} ± {std_f1:.4f}"
)

print(
    f"  Kappa    = {mean_kappa:.4f} ± {std_kappa:.4f}"
)

print("=" * 75)

CONVORELENET TRANSFER-LEARNING MASTER PIPELINE
Device: mps

[1/12] Loading EEG bundle...
X shape       : (5184, 22, 1000)
y shape       : (5184,)
Subjects      : ['1' '2' '3' '4' '5' '6' '7' '8' '9']
Classes       : [0 1 2 3]
Class names   : ['0', '1', '2', '3']
Sampling rate : 250
Calibration   : 0train
Test session  : 1test

Subjects used:
['1', '2', '3', '4', '5', '6', '7', '8', '9']


STARTING FULL LOSO

Fold 1/9


TARGET SUBJECT: 1

Preparing target subject 1
    Filtering 8-30 Hz...
    Fitting training-only normalization...
    Source: (4608, 22, 1000)
    Calibration: (288, 22, 1000)
    Test: (288, 22, 1000)

[2/12] Creating source model...
Source model:
Trainable parameters: 2,254,748
Total parameters    : 2,254,748

[3/12] Source-subject pretraining...
      pretrain           epoch   1/150 | loss=1.3934 | acc= 25.41% | lr=1.00e-03
      pretrain           epoch  10/150 | loss=1.3372 | acc= 34.09% | lr=9.89e-04
      pretrain           epoch  20/150 | loss=1.2787 | acc= 37.7

KeyboardInterrupt: 